# EPANET-AI: Очистка нулевых строк + расчёт производных показателей (скорость, этажность, свободный напор, Re, мощность насосов)

Этот ноутбук работает **полностью автономно по клеточкам (без консоли, PowerShell и терминальных команд)**.
Положите `EPANET_AI_Pipeline.ipynb` в корень папки **`EPANET-AI`** (рядом с папкой `data`) и нажмите **Run All** (или запускайте клетки по очереди `Shift + Enter`).

### Что делает ноутбук и какие новые файлы он создаёт:
1. **Клетки 1–5 — Подключение и аудит всех 13 CSV-таблиц по каждому городу:** проверка нулевых значений (`0.0` отдельно от пустых `"" -> null`), префиксов `c/e/p/v/u/t`, 24-часовых графиков и петель `Начало == Конец`.
2. **Клетки 6–7 — Удаление нулевых строк в 4 целевых таблицах** и запись очищенных файлов в **`data_cleaned/<город>/`**:
   - `Поток Насос.csv` (удаляются полностью нулевые резервные насосы `R0..R1439 == 0`, циклические сохраняются);
   - `Потребители Узел.csv` (удаляются строки с `Потребление == 0` или пусто);
   - `Потребление Узел.csv` (удаляются полностью нулевые строки `R0..R1439 == 0`);
   - `Узел.csv` (удаляются промежуточные узлы, где `Требуемое давление` не указано / `<= 0`).
3. **Клетка 8 — Вычисление взаимно выводимых параметров (по вашим формулам, СП 31.13330.2021 и книге Walski et al.)** с сохранением **6 новых таблиц** в **`data_cleaned/<город>/расчетные_параметры/`**:
   - **`Узел_этажность_и_свободный_напор.csv`** — этажность дома ($n = 1 + (H_{\text{треб}} - 10)/4$), свободный напор $H_{\text{своб}} = \text{HGL} - Z$ (м), давление (бар), запас/дефицит напора и суточный водоразбор ($\text{м}^3/\text{сут}$);
   - **`Свободный напор Узел.csv`** — полная матрица `ID,R0..R1439` свободных напоров на конечных узлах;
   - **`Труба_скорости_и_режимы_Walski.csv`** — площадь сечения, объём воды в трубе ($\text{м}^3$), упрощённый индекс `Q / D`, точная физическая скорость $V = 4Q/(\pi D^2)$ (м/с, Walski Eq. 2.9), скоростной напор $V^2/(2g)$ (Eq. 2.12), число Рейнольдса $\text{Re}$ и режим течения (Table 2.1), время пробега воды $t = L/V$ (мин, Ch. 2.9), оценка по шкале Walski (p. 292), потери напора $\Delta H$ и уклон $i$ (м/км);
   - **`Скорость Труба.csv`** — полная знаковая матрица `ID,R0..R1439` скоростей потока (м/с) по всем трубам;
   - **`Насос_рабочие_точки_и_мощность.csv`** — моточасы, число пусков, интерполяция напора $TDH(Q)$ по `Кривая QH` (Walski p. 58), полезная гидравлическая мощность (кВт) и суточная энергия (кВт·ч/сут, Walski p. 295);
   - **`График_потребления_анализ_24ч.csv`** — коэффициенты неравномерности $K_{\min}, K_{\text{сред}}, K_{\max}$ и час пика.
4. **Клетки 9–10 — Компактные сводки `summary_*.csv` / `.parquet` и сохранение итогового HTML/JSON отчёта.**

## Клетка 1. Настройки путей и параметров обработки
Укажите путь к вашей папке `data` (если ноутбук лежит в корне проекта `EPANET-AI`, ничего менять не нужно — папка `data` найдётся автоматически).

In [3]:
from pathlib import Path

# 1. Автоматический поиск корня проекта EPANET-AI и папки data
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Путь к исходным данным (можно указать конкретный путь, например Path(r"C:\EPANET-AI\data"))
DATA_DIR = PROJECT_ROOT / "data"

# Папка, в которую будут созданы НОВЫЕ очищенные CSV по каждому городу
OUTPUT_CLEANED_DIR = PROJECT_ROOT / "data_cleaned"

# Папка для компактных сводок (часовые срезы + min/max/mean + Parquet) и HTML/JSON отчётов
OUTPUT_SUMMARY_DIR = PROJECT_ROOT / "artifacts_epanet_ai"

# Настройки фильтрации:
CLEAN_IN_PLACE = False         # False = создавать новые файлы в OUTPUT_CLEANED_DIR (безопасно!); True = перезаписать исходные файлы в DATA_DIR
PRUNE_NODE_HEAD_TOO = False    # True = дополнительно удалять транзитные узлы без Требуемое давление также из "Напор Узел.csv"
EXPORT_PARQUET_SUMMARIES = True  # Создавать компактные таблицы сводок по широким рядам R0..R1439
COMPUTE_DERIVED_WALSKI_TABLES = True    # Рассчитывать и сохранять скорость, этажность, свободный напор, Re, мощность насосов
SAVE_WIDE_VELOCITY_AND_PRESSURE = True  # Записывать полные матрицы 'Скорость Труба.csv' и 'Свободный напор Узел.csv' (R0..R1439)  # Создавать компактные таблицы сводок по широким рядам R0..R1439

print(f"Рабочая папка проекта : {PROJECT_ROOT.resolve()}")
print(f"Папка исходных данных : {DATA_DIR.resolve()} (существует: {DATA_DIR.exists()})")
print(f"Папка новых CSV       : {OUTPUT_CLEANED_DIR.resolve()}")
print(f"Папка сводок и отчётов: {OUTPUT_SUMMARY_DIR.resolve()}")

Рабочая папка проекта : D:\Zhenechka\code\EPANET_AI
Папка исходных данных : D:\Zhenechka\code\EPANET_AI\data (существует: True)
Папка новых CSV       : D:\Zhenechka\code\EPANET_AI\data_cleaned
Папка сводок и отчётов: D:\Zhenechka\code\EPANET_AI\artifacts_epanet_ai


## Клетка 2. Подключение модулей и контрактов таблиц AquaBit / EPANET-AI
Подключаем необходимые библиотеки Python и при наличии папки `src/` автоматически подключаем внутренние модули проекта (ноутбук полностью работоспособен даже без папки `src/`).

In [4]:
import csv
import html
import io
import json
import math
import shutil
import sys
import time
import unicodedata
from collections import Counter
from dataclasses import asdict, dataclass, field as dc_field
from typing import Any

import numpy as np
import pandas as pd
try:
    from IPython.display import HTML, Markdown, display
except ImportError:
    HTML = Markdown = lambda x: x
    display = print

# Опционально подключаем pyarrow для ускоренного C++-чтения матриц R0..R1439 и записи Zstd Parquet
try:
    import pyarrow as pa
    import pyarrow.csv as pacsv
    import pyarrow.parquet as pq
    HAS_PYARROW = True
except ImportError:
    HAS_PYARROW = False

# Если рядом есть папка src/ из репозитория EPANET-AI, подключаем её в sys.path без консольных команд
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_DIR.resolve()))

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 1000)

# Токены пропусков (Принцип 6: пустая строка "" -> null, это НЕ 0.0!)
NULL_STRINGS = {"", "null", "NULL", "None"}
IGNORED_FILES = {".DS_Store", "Thumbs.db", "desktop.ini", "pruned_zero_manifest.json"}

# Нормализация имён 13 CSV-таблиц папки города
def normalize_stem(filename: str) -> str:
    stem = unicodedata.normalize("NFC", Path(filename).stem).strip().lower()
    stem = stem.replace("ё", "е").replace("_", " ").replace("-", " ")
    return " ".join(stem.split())

CITY_TABLE_SPECS: dict[str, dict[str, Any]] = {
    "узел": {
        "canonical": "nodes.csv",
        "is_wide": False,
        "is_clean_target": True,
        "id_col": "ID",
        "expected_prefixes": ("c",),
        "description": "Справочник узлов сети (геодезическая отметка и требуемое давление)",
    },
    "труба": {
        "canonical": "pipes.csv",
        "is_wide": False,
        "is_clean_target": False,
        "id_col": "ID",
        "expected_prefixes": ("e",),
        "description": "Справочник участков трубопроводов (диаметр, длина, шероховатость)",
    },
    "источники": {
        "canonical": "sources.csv",
        "is_wide": False,
        "is_clean_target": False,
        "id_col": "ID",
        "expected_prefixes": ("e", "c"),
        "description": "Источники питания / граничные узлы фиксированного напора",
    },
    "источник": {
        "canonical": "sources.csv",
        "is_wide": False,
        "is_clean_target": False,
        "id_col": "ID",
        "expected_prefixes": ("e", "c"),
        "description": "Источники питания / граничные узлы фиксированного напора",
    },
    "клапан": {
        "canonical": "valves.csv",
        "is_wide": False,
        "is_clean_target": False,
        "id_col": "ID",
        "expected_prefixes": ("v",),
        "description": "Запорно-регулирующая арматура и регуляторы",
    },
    "насос": {
        "canonical": "pumps.csv",
        "is_wide": False,
        "is_clean_target": False,
        "id_col": "ID",
        "expected_prefixes": ("p",),
        "description": "Насосные агрегаты и ссылки на напорно-расходные кривые u*",
    },
    "кривая qh": {
        "canonical": "pump_curves.csv",
        "is_wide": False,
        "is_clean_target": False,
        "id_col": "ID",
        "expected_prefixes": ("u",),
        "description": "Паспортные кривые насосов Q-H (точка X=0 — штатный напор холостого хода H0)",
    },
    "потребители узел": {
        "canonical": "demands.csv",
        "is_wide": False,
        "is_clean_target": True,
        "id_col": "ID",
        "expected_prefixes": ("c",),
        "description": "Базовые нагрузки абонентов на узлах и привязка к суточным графикам t*",
    },
    "график потребления": {
        "canonical": "irregularity_curves.csv",
        "is_wide": False,
        "is_clean_target": False,
        "id_col": "ID",
        "expected_prefixes": ("t",),
        "description": "Суточные графики часовой неравномерности (строго по 24 строки на каждый t* ID)",
    },
    "напор узел": {
        "canonical": "wide/node_head.csv",
        "is_wide": True,
        "is_clean_target": False,
        "id_col": "ID",
        "expected_prefixes": ("c", "e"),
        "description": "Широкая матрица полного пьезометрического напора HGL (R0..R1439)",
    },
    "потребление узел": {
        "canonical": "wide/node_demand.csv",
        "is_wide": True,
        "is_clean_target": True,
        "id_col": "ID",
        "expected_prefixes": ("c",),
        "description": "Широкая матрица узлового водоразбора по шагам R0..R1439",
    },
    "поток труба": {
        "canonical": "wide/pipe_flow.csv",
        "is_wide": True,
        "is_clean_target": False,
        "id_col": "ID",
        "expected_prefixes": ("e",),
        "description": "Широкая матрица расхода по трубам R0..R1439 (знак '-' означает обратный ток Конец -> Начало)",
    },
    "поток клапан": {
        "canonical": "wide/valve_flow.csv",
        "is_wide": True,
        "is_clean_target": False,
        "id_col": "ID",
        "expected_prefixes": ("v",),
        "description": "Широкая матрица расхода через клапаны R0..R1439",
    },
    "поток насос": {
        "canonical": "wide/pump_flow.csv",
        "is_wide": True,
        "is_clean_target": True,
        "id_col": "ID",
        "expected_prefixes": ("p",),
        "description": "Широкая матрица подачи насосов R0..R1439 (0.00 на всех шагах = резервный/выключенный насос)",
    },
}

print(f"Модули успешно подключены! Быстрый движок PyArrow доступен: {HAS_PYARROW}")

Модули успешно подключены! Быстрый движок PyArrow доступен: True


## Клетка 3. Обнаружение папок городов и подключение CSV-файлов
Клетка автоматически находит все города внутри `DATA_DIR` (или работает с одной папкой города, если `DATA_DIR` указывает прямо на неё). Если у вас ещё нет папки `data` рядом с ноутбуком, при желании можно включить `CREATE_DEMO_IF_DATA_MISSING = True` для тестового прогона.

In [5]:
CREATE_DEMO_IF_DATA_MISSING = False  # Поставьте True, если хотите сгенерировать демо-папку data/тенино для проверки ноутбука

def detect_encoding_and_delimiter(file_path: Path) -> tuple[str, bool, str]:
    """Определяет кодировку (utf-8 / utf-8-sig / cp1251) и разделитель CSV по первым 16 КБ файла."""
    with file_path.open("rb") as f:
        sample_bytes = f.read(16384)
    has_bom = sample_bytes.startswith(b"\xef\xbb\xbf")
    if has_bom:
        enc = "utf-8-sig"
        sample_text = sample_bytes.decode("utf-8-sig", errors="replace")
    else:
        try:
            sample_text = sample_bytes.decode("utf-8")
            enc = "utf-8"
        except UnicodeDecodeError:
            enc = "cp1251"
            sample_text = sample_bytes.decode("cp1251", errors="replace")
    first_line = sample_text.splitlines()[0] if sample_text.splitlines() else ""
    counts = {",": first_line.count(","), ";": first_line.count(";"), "\t": first_line.count("\t")}
    delim = max(counts, key=counts.get) if max(counts.values()) > 0 else ","
    return enc, has_bom, delim

def discover_cities(root_dir: Path) -> list[Path]:
    """Находит все папки городов с CSV-таблицами AquaBit."""
    if not root_dir.exists():
        return []
    direct_csvs = [p for p in root_dir.iterdir() if p.is_file() and normalize_stem(p.name) in CITY_TABLE_SPECS]
    if len(direct_csvs) >= 2:
        return [root_dir]
    found = []
    for sub in sorted(root_dir.iterdir(), key=lambda p: unicodedata.normalize("NFC", p.name).lower()):
        if sub.is_dir() and not sub.name.startswith("."):
            sub_csvs = [p for p in sub.iterdir() if p.is_file() and normalize_stem(p.name) in CITY_TABLE_SPECS]
            if len(sub_csvs) >= 2:
                found.append(sub)
    return found

if not DATA_DIR.exists() and CREATE_DEMO_IF_DATA_MISSING:
    demo_city = DATA_DIR / "тенино_демо"
    demo_city.mkdir(parents=True, exist_ok=True)
    r_cols = ",".join(["ID"] + [f"R{i}" for i in range(1440)])
    z_1440 = ",".join(["0.00"] * 1440)
    (demo_city / "Узел.csv").write_text("ID,Высота,Требуемое давление\nc171,0.00,18\nc172,118.50,14\nc173,119.00,0\nc174,120.10,\n", encoding="utf-8")
    (demo_city / "Источники.csv").write_text("ID,Расчетный напор\ne701,150.0\n", encoding="utf-8")
    (demo_city / "Труба.csv").write_text("ID,Начало,Конец,Диаметр,Длина,Шероховатость,Малые потери\ne702,e701,c171,200,120.0,15.0,0\ne703,c171,c172,150,85.0,15.0,0\n", encoding="utf-8")
    (demo_city / "Насос.csv").write_text("ID,Начало,Конец,Кривая насоса\np1263,c171,c172,u123\np1264,c172,c173,u123\n", encoding="utf-8")
    (demo_city / "Клапан.csv").write_text("ID,Начало,Конец,Тип,Настройка\nv1284,c172,c173,V1,50\n", encoding="utf-8")
    (demo_city / "Кривая QH.csv").write_text("ID,X,Y\nu123,0,64\nu123,1800,54\nu123,3600,38\n", encoding="utf-8")
    (demo_city / "График потребления.csv").write_text("ID,Данные\n" + "\n".join("t104,1" for _ in range(24)) + "\n", encoding="utf-8")
    (demo_city / "Потребители Узел.csv").write_text("ID,Потребление,График\nc171,45.0,t104\nc172,12.5,t104\nc173,0.0,t104\n", encoding="utf-8")
    (demo_city / "Напор Узел.csv").write_text("\n".join([r_cols, "c171," + ",".join(["145.20"]*1440), "c172," + ",".join(["141.10"]*1440), "c173," + ",".join(["139.00"]*1440), "c174," + ",".join(["138.00"]*1440)]) + "\n", encoding="utf-8")
    (demo_city / "Потребление Узел.csv").write_text("\n".join([r_cols, "c171," + ",".join(["45.00"]*1440), "c172," + ",".join(["12.50"]*1440), f"c173,{z_1440}", f"c174,{z_1440}"]) + "\n", encoding="utf-8")
    (demo_city / "Поток Труба.csv").write_text("\n".join([r_cols, "e702," + ",".join(["57.50"]*1440), "e703," + ",".join(["-12.50"]*1440)]) + "\n", encoding="utf-8")
    (demo_city / "Поток Клапан.csv").write_text("\n".join([r_cols, "v1284," + ",".join(["12.50"]*1440)]) + "\n", encoding="utf-8")
    (demo_city / "Поток Насос.csv").write_text("\n".join([r_cols, f"p1263,{z_1440}", "p1264," + ",".join(["0.00"]*360 + ["120.00"]*1080)]) + "\n", encoding="utf-8")

CITY_DIRS = discover_cities(DATA_DIR)
if not CITY_DIRS:
    print(f"⚠️ Внимание: по пути {DATA_DIR.resolve()} пока не найдено папок городов с CSV-файлами.")
    print("Положите ваши папки городов (тенино, хорезмская, Орша, модель семей, куляб, Алматы) внутрь папки data/ и перезапустите эту клетку.")
else:
    inventory_rows = []
    for cdir in CITY_DIRS:
        csv_files = [p for p in sorted(cdir.iterdir()) if p.is_file() and p.suffix.lower() == ".csv" and p.name not in IGNORED_FILES]
        total_mb = sum(p.stat().st_size for p in csv_files) / (1024 * 1024)
        sample_enc, _, sample_delim = detect_encoding_and_delimiter(csv_files[0]) if csv_files else ("utf-8", False, ",")
        inventory_rows.append({
            "Город": unicodedata.normalize("NFC", cdir.name),
            "Кол-во CSV": len(csv_files),
            "Объём (МБ)": round(total_mb, 2),
            "Кодировка": sample_enc,
            "Разделитель": repr(sample_delim),
            "Путь": str(cdir),
        })
    print(f"✅ Найдено папок городов: {len(CITY_DIRS)}")
    display(pd.DataFrame(inventory_rows))

✅ Найдено папок городов: 6


,Город,Кол-во CSV,Объём (МБ),Кодировка,Разделитель,Путь
0,Алматы,11,2173.32,utf-8-sig,"','",d:\Zhenechka\code\EPANET_AI\data\Алматы
1,куляб,14,37.72,utf-8-sig,"','",d:\Zhenechka\code\EPANET_AI\data\куляб
2,модель семей,14,619.23,utf-8-sig,"','",d:\Zhenechka\code\EPANET_AI\data\модель семей
3,Орша,14,252.95,utf-8-sig,"','",d:\Zhenechka\code\EPANET_AI\data\Орша
4,тенино,14,16.30,utf-8-sig,"','",d:\Zhenechka\code\EPANET_AI\data\тенино
5,хорезмская,14,297.49,utf-8-sig,"','",d:\Zhenechka\code\EPANET_AI\data\хорезмская


## Клетка 4. Функции проверки нулевых значений, префиксов ID, 24-часовых графиков и этажности
Определяем быстрые векторизованные функции чтения матриц `R0..R1439` и статических справочников:
- строгое разделение `0.0` и пустых ячеек `"" -> null` (**Принцип 6**);
- формула требуемого напора по этажности $H_{\text{треб}} = 10 + 4(n - 1)$ м (СП 31.13330.2021);
- контроль 24 строк на каждый график `t*` и префиксов `c, e, p, v, u, t`.

In [6]:
def infer_floors_label(pressure_m: float) -> str:
    """Переводит требуемое давление (м) в этажность застройки по формуле H = 10 + 4*(n - 1) м."""
    if pressure_m <= 0.0:
        return "0 м (транзитный узел)"
    if pressure_m < 10.0:
        return f"<10 м ({pressure_m:g} м — ниже нормы 1 эт.!)"
    diff = pressure_m - 10.0
    if abs(diff - round(diff / 4.0) * 4.0) < 1e-6:
        floors = 1 + int(round(diff / 4.0))
        return f"{floors} эт. ({pressure_m:g} м)"
    return f"нестанд. {pressure_m:g} м (~{1.0 + diff / 4.0:.1f} эт.)"


def load_wide_matrix_fast(file_path: Path) -> tuple[list[str], list[str], np.ndarray, np.ndarray, str, bool]:
    """Читает широкую матрицу ID,R0..R1439 и возвращает (ids, r_cols, float_matrix, null_mask, encoding, has_bom)."""
    enc, has_bom, delim = detect_encoding_and_delimiter(file_path)
    if HAS_PYARROW:
        pa_enc = "utf8" if enc in {"utf-8", "utf-8-sig"} else enc
        with file_path.open("rb") as f:
            first_bytes = f.read(65536)
        hdr_line = first_bytes.decode(enc, errors="replace").splitlines()[0]
        raw_headers = [c.strip() for c in next(csv.reader([hdr_line], delimiter=delim))]
        col_types = {raw_headers[0]: pa.string()}
        for h in raw_headers[1:]:
            col_types[h] = pa.float64()
        tbl = pacsv.read_csv(
            file_path,
            read_options=pacsv.ReadOptions(encoding=pa_enc),
            parse_options=pacsv.ParseOptions(delimiter=delim),
            convert_options=pacsv.ConvertOptions(
                column_types=col_types,
                strings_can_be_null=True,
                null_values=["", "null", "NULL", "None"],
            ),
        )
        ids = ["" if v is None else str(v).strip() for v in tbl.column(0).to_pylist()]
        r_cols = raw_headers[1:]
        n_rows = tbl.num_rows
        n_steps = len(r_cols)
        mat = np.empty((n_rows, n_steps), dtype=np.float64)
        null_mask = np.zeros((n_rows, n_steps), dtype=bool)
        for j in range(n_steps):
            col = tbl.column(j + 1).combine_chunks()
            if col.null_count > 0:
                null_mask[:, j] = col.is_null().to_numpy(zero_copy_only=False)
                mat[:, j] = col.fill_null(np.nan).to_numpy(zero_copy_only=False)
            else:
                mat[:, j] = col.to_numpy(zero_copy_only=False)
        return ids, r_cols, mat, null_mask, enc, has_bom

    # Резервная ветка на чистом pandas/numpy (если pyarrow не установлен)
    df = pd.read_csv(
        file_path,
        sep=delim,
        encoding=enc,
        dtype={0: str},
        keep_default_na=False,
        low_memory=False,
    )
    ids = [str(v).strip() for v in df.iloc[:, 0].tolist()]
    r_cols = list(df.columns[1:])
    raw_vals = df.iloc[:, 1:]
    null_mask = raw_vals.isin(NULL_STRINGS).to_numpy(dtype=bool)
    num_df = raw_vals.apply(pd.to_numeric, errors="coerce")
    mat = num_df.to_numpy(dtype=np.float64)
    return ids, r_cols, mat, null_mask, enc, has_bom


def parse_cell_num(cell: str) -> tuple[str, float]:
    s = cell.strip()
    if s in NULL_STRINGS:
        return ("null", 0.0)
    try:
        v = float(s)
        if math.isnan(v):
            return ("nan", 0.0)
        return ("num", v)
    except ValueError:
        return ("str", 0.0)


def inspect_city_tables(city_dir: Path) -> dict[str, Any]:
    """Выполняет полный аудит всех таблиц одного города без изменения файлов."""
    t0 = time.perf_counter()
    city_name = unicodedata.normalize("NFC", city_dir.name)
    files = sorted(
        [p for p in city_dir.iterdir() if p.is_file() and p.suffix.lower() == ".csv" and p.name not in IGNORED_FILES],
        key=lambda p: (0 if normalize_stem(p.name) == "узел" else 1, normalize_stem(p.name)),
    )

    table_rows: list[dict[str, Any]] = []
    floor_counter: Counter[str] = Counter()
    non_24_schedules: dict[str, int] = {}
    self_loop_links = 0
    prefix_violations = 0
    transit_node_ids: list[str] = []
    standby_pump_ids: list[str] = []
    zero_consumer_rows = 0
    zero_demand_matrix_rows = 0

    for fpath in files:
        stem = normalize_stem(fpath.name)
        spec = CITY_TABLE_SPECS.get(stem, {
            "canonical": fpath.name,
            "is_wide": False,
            "is_clean_target": False,
            "expected_prefixes": (),
            "description": "Дополнительная CSV-таблица",
        })
        size_kb = round(fpath.stat().st_size / 1024.0, 1)
        expected_pfxs = spec.get("expected_prefixes", ())

        if spec["is_wide"]:
            ids, r_cols, mat, null_mask, _, _ = load_wide_matrix_fast(fpath)
            n_rows, n_steps = mat.shape
            for oid in ids:
                if oid and expected_pfxs and not any(oid.startswith(p) for p in expected_pfxs):
                    prefix_violations += 1

            ieee_nan = (~null_mask) & np.isnan(mat)
            valid = (~null_mask) & (~ieee_nan)
            zero_mask = valid & (mat == 0.0)

            null_cells = int(np.count_nonzero(null_mask))
            zero_cells = int(np.count_nonzero(zero_mask))
            if n_rows > 0 and n_steps > 0:
                all_zero_mask = (np.sum(valid, axis=1) > 0) & np.all(zero_mask | (~valid), axis=1) & (np.sum(null_mask, axis=1) == 0)
                partial_zero_mask = (~all_zero_mask) & np.any(zero_mask, axis=1) & np.any(valid & (mat != 0.0), axis=1)
                neg_rows = int(np.count_nonzero(np.any(valid & (mat < 0.0), axis=1)))
            else:
                all_zero_mask = np.zeros(n_rows, dtype=bool)
                partial_zero_mask = np.zeros(n_rows, dtype=bool)
                neg_rows = 0

            all_zero_n = int(np.count_nonzero(all_zero_mask))
            partial_zero_n = int(np.count_nonzero(partial_zero_mask))

            if spec["canonical"] == "wide/pump_flow.csv":
                standby_pump_ids = [ids[i] for i in np.flatnonzero(all_zero_mask)]
            elif spec["canonical"] == "wide/node_demand.csv":
                zero_demand_matrix_rows = all_zero_n

            note_parts = []
            if spec["is_clean_target"]:
                note_parts.append(f"К удалению: {all_zero_n} полностью 0-строк")
            if partial_zero_n > 0:
                note_parts.append(f"Сохраняются {partial_zero_n} строк с частичными нулями")
            if neg_rows > 0 and spec["canonical"] == "wide/pipe_flow.csv":
                note_parts.append(f"Обратный поток (Q<0): {neg_rows} труб")

            table_rows.append({
                "Файл": fpath.name,
                "Тип": f"Матрица R0..R{max(0, n_steps-1)}",
                "Целевая очистка": "ДА" if spec["is_clean_target"] else "нет",
                "Строк": n_rows,
                "К удалению (0-строк)": all_zero_n if spec["is_clean_target"] else 0,
                "Полностью 0-строк": f"{all_zero_n} ({(100*all_zero_n/n_rows if n_rows else 0):.1f}%)",
                "Частично 0-строк": partial_zero_n,
                "Пустых null": null_cells,
                "Размер (КБ)": size_kb,
                "Примечание": "; ".join(note_parts) if note_parts else "Норма",
            })
        else:
            enc, _, delim = detect_encoding_and_delimiter(fpath)
            text = fpath.read_text(encoding=enc, errors="replace")
            rows = [r for r in csv.reader(io.StringIO(text), delimiter=delim) if any(c.strip() for c in r)]
            headers = [h.strip() for h in rows[0]] if rows else []
            data_rows = rows[1:] if len(rows) > 1 else []
            n_rows = len(data_rows)

            for r in data_rows:
                oid = r[0].strip() if r else ""
                if oid and expected_pfxs and not any(oid.startswith(p) for p in expected_pfxs):
                    prefix_violations += 1

            # Подсчёт нулей и пропусков по ячейкам
            z_cells = 0
            n_cells = 0
            for r in data_rows:
                for c in r:
                    k, v = parse_cell_num(c)
                    if k == "null":
                        n_cells += 1
                    elif k == "num" and v == 0.0:
                        z_cells += 1

            # Проверка петель Начало == Конец в трубах, насосах, клапанах
            if "Начало" in headers and "Конец" in headers:
                i1, i2 = headers.index("Начало"), headers.index("Конец")
                for r in data_rows:
                    if len(r) > max(i1, i2) and r[i1].strip() and r[i1].strip() == r[i2].strip():
                        self_loop_links += 1

            drop_cnt = 0
            note_parts = []
            if spec["canonical"] == "nodes.csv":
                req_idx = next((i for i, h in enumerate(headers) if "давлен" in h.lower()), None)
                if req_idx is not None:
                    for r in data_rows:
                        nid = r[0].strip() if r else ""
                        cell = r[req_idx] if req_idx < len(r) else ""
                        k, v = parse_cell_num(cell)
                        if k == "null" or (k == "num" and v <= 0.0):
                            drop_cnt += 1
                            transit_node_ids.append(nid)
                        elif k == "num":
                            floor_counter[infer_floors_label(v)] += 1
                note_parts.append(f"Транзитных узлов без Hтреб к удалению: {drop_cnt}")

            elif spec["canonical"] == "demands.csv":
                q_idx = next((i for i, h in enumerate(headers) if "потреблен" in h.lower()), None)
                if q_idx is not None:
                    for r in data_rows:
                        cell = r[q_idx] if q_idx < len(r) else ""
                        k, v = parse_cell_num(cell)
                        if k == "null" or (k == "num" and v == 0.0):
                            drop_cnt += 1
                zero_consumer_rows = drop_cnt
                note_parts.append(f"Абонентов с Потребление=0 к удалению: {drop_cnt}")

            elif spec["canonical"] == "irregularity_curves.csv":
                cnts = Counter(r[0].strip() for r in data_rows if r and r[0].strip() not in NULL_STRINGS)
                non_24 = {cid: c for cid, c in cnts.items() if c != 24}
                non_24_schedules.update(non_24)
                note_parts.append("Все графики строго по 24 ч" if not non_24 else f"⚠️ Графиков != 24 ч: {len(non_24)}")

            elif spec["canonical"] == "pump_curves.csv":
                q0_pts = sum(1 for r in data_rows if len(r) > 1 and parse_cell_num(r[1]) == ("num", 0.0))
                if q0_pts > 0:
                    note_parts.append(f"Точек X=0 (напор холостого хода H0): {q0_pts} — НЕ удаляются")

            table_rows.append({
                "Файл": fpath.name,
                "Тип": "Статическая таблица",
                "Целевая очистка": "ДА" if spec["is_clean_target"] else "нет",
                "Строк": n_rows,
                "К удалению (0-строк)": drop_cnt,
                "Полностью 0-строк": f"{drop_cnt} ({(100*drop_cnt/n_rows if n_rows else 0):.1f}%)" if spec["is_clean_target"] else "—",
                "Частично 0-строк": z_cells,
                "Пустых null": n_cells,
                "Размер (КБ)": size_kb,
                "Примечание": "; ".join(note_parts) if note_parts else "Норма",
            })

    elapsed_ms = round((time.perf_counter() - t0) * 1000.0, 1)
    return {
        "city_name": city_name,
        "city_dir": city_dir,
        "elapsed_ms": elapsed_ms,
        "tables_df": pd.DataFrame(table_rows),
        "floor_distribution": dict(floor_counter),
        "non_24_schedules": non_24_schedules,
        "self_loop_links": self_loop_links,
        "prefix_violations": prefix_violations,
        "transit_node_ids": transit_node_ids,
        "standby_pump_ids": standby_pump_ids,
        "zero_consumer_rows": zero_consumer_rows,
        "zero_demand_matrix_rows": zero_demand_matrix_rows,
    }

print("✅ Функции проверки нулей и доменной валидации готовы.")

✅ Функции проверки нулей и доменной валидации готовы.


## Клетка 5. Запуск проверки всех файлов на нулевые значения (без перезаписи файлов)
Выводит подробные таблицы по каждому городу: где именно находятся нули, сколько строк попадёт под очистку в 4 целевых таблицах и как распределены конечные узлы по этажности застройки.

In [ ]:
AUDIT_RESULTS: list[dict[str, Any]] = []

for cdir in CITY_DIRS:
    res = inspect_city_tables(cdir)
    AUDIT_RESULTS.append(res)
    display(Markdown(
        f"### Город: `{res['city_name']}` (проверено за {res['elapsed_ms']} мс)\n"
        f"- Узлов без `Требуемое давление` (`Узел.csv`): **{len(res['transit_node_ids'])}** | "
        f"Резервных насосов `R0..R1439 == 0` (`Поток Насос.csv`): **{len(res['standby_pump_ids'])}** | "
        f"Нулевых абонентов (`Потребители Узел.csv`): **{res['zero_consumer_rows']}** | "
        f"Нулевых строк `Потребление Узел.csv`: **{res['zero_demand_matrix_rows']}**\n"
        f"- Нарушений префиксов `c/e/p/v/u/t`: **{res['prefix_violations']}** | "
        f"Графиков `!= 24 ч`: **{len(res['non_24_schedules'])}** | "
        f"Топологических петель `Начало == Конец`: **{res['self_loop_links']}**"
    ))
    display(res["tables_df"])
    if res["floor_distribution"]:
        fl_df = pd.DataFrame(
            [{"Этажность (Hтреб = 10 + 4*(n-1) м)": k, "Кол-во конечных узлов": v}
             for k, v in sorted(res["floor_distribution"].items())]
        )
        display(fl_df)

## Клетка 6. Функции удаления нулевых строк и создания новых очищенных файлов
Удаление строк выполняется **потоково по булевой маске строк**: все оставшиеся строки записываются в новые файлы **байт-в-байт в исходном формате AquaBit** (без округления чисел с плавающей точкой и без изменения строковых `ID`).

Целевые таблицы удаления нулевых строк:
1. **`Поток Насос.csv`** — удаляются строки, где все `R0..R1439 == 0`;
2. **`Потребители Узел.csv`** — удаляются строки, где `Потребление == 0` или не указано;
3. **`Потребление Узел.csv`** — удаляются строки, где все `R0..R1439 == 0`;
4. **`Узел.csv`** — удаляются строки, где `Требуемое давление` не указано (`""` или `<= 0`).

In [7]:
def filter_csv_lines_lossless(
    src_file: Path,
    dst_file: Path,
    keep_mask: list[bool] | np.ndarray,
    encoding: str,
    has_bom: bool,
) -> int:
    """Создаёт новый CSV-файл, сохраняя заголовок и только строки с keep_mask[i] == True без переформатирования."""
    dst_file.parent.mkdir(parents=True, exist_ok=True)
    read_enc = "utf-8-sig" if (encoding in {"utf-8", "utf-8-sig"} and has_bom) else encoding
    write_enc = "utf-8-sig" if has_bom else ("utf-8" if encoding == "utf-8-sig" else encoding)

    target_tmp = dst_file.with_suffix(".csv.tmp_epanet") if src_file.resolve() == dst_file.resolve() else dst_file
    with src_file.open("r", encoding=read_enc, newline="") as fin, target_tmp.open("w", encoding=write_enc, newline="") as fout:
        header_done = False
        idx = 0
        n_mask = len(keep_mask)
        for line in fin:
            if not line.strip():
                continue
            if not header_done:
                fout.write(line)
                header_done = True
                continue
            if idx < n_mask and bool(keep_mask[idx]):
                fout.write(line)
            idx += 1

    if target_tmp != dst_file:
        target_tmp.replace(dst_file)
    return dst_file.stat().st_size


def clean_city_zero_rows(
    city_dir: Path,
    dst_city_dir: Path,
    *,
    prune_node_head: bool = False,
) -> dict[str, Any]:
    """Создаёт новую папку города с очищенными от нулевых строк 4 таблицами и копией остальных файлов."""
    t0 = time.perf_counter()
    dst_city_dir.mkdir(parents=True, exist_ok=True)
    city_name = unicodedata.normalize("NFC", city_dir.name)

    all_files = sorted(
        [p for p in city_dir.iterdir() if p.is_file() and p.name not in IGNORED_FILES],
        key=lambda p: (0 if normalize_stem(p.name) == "узел" else 1, normalize_stem(p.name)),
    )

    pruned_transit_nodes: list[str] = []
    pruned_standby_pumps: list[str] = []
    pruned_zero_consumers = 0
    pruned_zero_demand_rows = 0
    file_stats: list[dict[str, Any]] = []

    for src_file in all_files:
        dst_file = dst_city_dir / src_file.name
        size_before = src_file.stat().st_size

        if src_file.suffix.lower() != ".csv":
            if src_file.resolve() != dst_file.resolve():
                shutil.copy2(src_file, dst_file)
            continue

        stem = normalize_stem(src_file.name)
        spec = CITY_TABLE_SPECS.get(stem)
        if spec is None:
            if src_file.resolve() != dst_file.resolve():
                shutil.copy2(src_file, dst_file)
            continue

        # 1. Широкие матрицы R0..R1439
        if spec["is_wide"]:
            is_target = spec["canonical"] in {"wide/pump_flow.csv", "wide/node_demand.csv"} or (
                prune_node_head and spec["canonical"] == "wide/node_head.csv"
            )
            if is_target:
                ids, _, mat, null_mask, enc, has_bom = load_wide_matrix_fast(src_file)
                n_rows = len(ids)
                if spec["canonical"] == "wide/node_head.csv" and prune_node_head:
                    drop_set = set(pruned_transit_nodes)
                    drop_mask = np.array([oid in drop_set for oid in ids], dtype=bool)
                else:
                    ieee_nan = (~null_mask) & np.isnan(mat)
                    valid = (~null_mask) & (~ieee_nan)
                    zero_mask = valid & (mat == 0.0)
                    drop_mask = (np.sum(valid, axis=1) > 0) & np.all(zero_mask | (~valid), axis=1) & (np.sum(null_mask, axis=1) == 0)

                removed_n = int(np.count_nonzero(drop_mask))
                if spec["canonical"] == "wide/pump_flow.csv":
                    pruned_standby_pumps = [ids[i] for i in np.flatnonzero(drop_mask)]
                elif spec["canonical"] == "wide/node_demand.csv":
                    pruned_zero_demand_rows = removed_n

                if removed_n > 0:
                    size_after = filter_csv_lines_lossless(src_file, dst_file, ~drop_mask, enc, has_bom)
                else:
                    if src_file.resolve() != dst_file.resolve():
                        shutil.copy2(src_file, dst_file)
                    size_after = dst_file.stat().st_size

                file_stats.append({
                    "Город": city_name,
                    "Таблица": src_file.name,
                    "Действие": "Очищена от 0-строк",
                    "Строк ДО": n_rows,
                    "Строк ПОСЛЕ": n_rows - removed_n,
                    "Удалено строк": removed_n,
                    "КБ ДО": round(size_before / 1024.0, 1),
                    "КБ ПОСЛЕ": round(size_after / 1024.0, 1),
                })
            else:
                if src_file.resolve() != dst_file.resolve():
                    shutil.copy2(src_file, dst_file)
                ids, _, _, _, _, _ = load_wide_matrix_fast(src_file)
                file_stats.append({
                    "Город": city_name,
                    "Таблица": src_file.name,
                    "Действие": "Скопирована без изменений",
                    "Строк ДО": len(ids),
                    "Строк ПОСЛЕ": len(ids),
                    "Удалено строк": 0,
                    "КБ ДО": round(size_before / 1024.0, 1),
                    "КБ ПОСЛЕ": round(dst_file.stat().st_size / 1024.0, 1),
                })

        # 2. Статические таблицы
        else:
            enc, has_bom, delim = detect_encoding_and_delimiter(src_file)
            text = src_file.read_text(encoding=enc, errors="replace")
            rows = [r for r in csv.reader(io.StringIO(text), delimiter=delim) if any(c.strip() for c in r)]
            headers = [h.strip() for h in rows[0]] if rows else []
            data_rows = rows[1:] if len(rows) > 1 else []
            n_rows = len(data_rows)

            if spec["canonical"] == "nodes.csv":
                req_idx = next((i for i, h in enumerate(headers) if "давлен" in h.lower()), None)
                drop_mask_list = []
                for r in data_rows:
                    nid = r[0].strip() if r else ""
                    cell = r[req_idx] if (req_idx is not None and req_idx < len(r)) else ""
                    k, v = parse_cell_num(cell)
                    is_zero_or_empty = (k == "null") or (k == "num" and v <= 0.0)
                    drop_mask_list.append(is_zero_or_empty)
                    if is_zero_or_empty:
                        pruned_transit_nodes.append(nid)
                removed_n = sum(drop_mask_list)
                keep_mask_list = [not d for d in drop_mask_list]
                if removed_n > 0:
                    filter_csv_lines_lossless(src_file, dst_file, keep_mask_list, enc, has_bom)
                elif src_file.resolve() != dst_file.resolve():
                    shutil.copy2(src_file, dst_file)
                file_stats.append({
                    "Город": city_name,
                    "Таблица": src_file.name,
                    "Действие": "Очищена (без Требуемое давление)",
                    "Строк ДО": n_rows,
                    "Строк ПОСЛЕ": n_rows - removed_n,
                    "Удалено строк": removed_n,
                    "КБ ДО": round(size_before / 1024.0, 1),
                    "КБ ПОСЛЕ": round(dst_file.stat().st_size / 1024.0, 1),
                })

            elif spec["canonical"] == "demands.csv":
                q_idx = next((i for i, h in enumerate(headers) if "потреблен" in h.lower()), None)
                drop_mask_list = []
                for r in data_rows:
                    cell = r[q_idx] if (q_idx is not None and q_idx < len(r)) else ""
                    k, v = parse_cell_num(cell)
                    drop_mask_list.append((k == "null") or (k == "num" and v == 0.0))
                removed_n = sum(drop_mask_list)
                pruned_zero_consumers = removed_n
                keep_mask_list = [not d for d in drop_mask_list]
                if removed_n > 0:
                    filter_csv_lines_lossless(src_file, dst_file, keep_mask_list, enc, has_bom)
                elif src_file.resolve() != dst_file.resolve():
                    shutil.copy2(src_file, dst_file)
                file_stats.append({
                    "Город": city_name,
                    "Таблица": src_file.name,
                    "Действие": "Очищена (Потребление == 0)",
                    "Строк ДО": n_rows,
                    "Строк ПОСЛЕ": n_rows - removed_n,
                    "Удалено строк": removed_n,
                    "КБ ДО": round(size_before / 1024.0, 1),
                    "КБ ПОСЛЕ": round(dst_file.stat().st_size / 1024.0, 1),
                })

            else:
                if src_file.resolve() != dst_file.resolve():
                    shutil.copy2(src_file, dst_file)
                file_stats.append({
                    "Город": city_name,
                    "Таблица": src_file.name,
                    "Действие": "Скопирована без изменений",
                    "Строк ДО": n_rows,
                    "Строк ПОСЛЕ": n_rows,
                    "Удалено строк": 0,
                    "КБ ДО": round(size_before / 1024.0, 1),
                    "КБ ПОСЛЕ": round(dst_file.stat().st_size / 1024.0, 1),
                })

    # Сохраняем манифест удалённых транзитных узлов и резервных насосов для ссылочной целостности
    manifest_path = dst_city_dir / "pruned_zero_manifest.json"
    manifest_data = {
        "city_name": city_name,
        "cleaned_tables": [
            "Поток Насос.csv",
            "Потребители Узел.csv",
            "Потребление Узел.csv",
            "Узел.csv",
        ] + (["Напор Узел.csv"] if prune_node_head else []),
        "pruned_transit_node_ids": pruned_transit_nodes,
        "pruned_standby_pump_ids": pruned_standby_pumps,
        "pruned_zero_consumers_count": pruned_zero_consumers,
        "pruned_zero_demand_matrix_rows_count": pruned_zero_demand_rows,
    }
    manifest_path.write_text(json.dumps(manifest_data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    kb_before = sum(r["КБ ДО"] for r in file_stats)
    kb_after = sum(r["КБ ПОСЛЕ"] for r in file_stats)
    return {
        "city_name": city_name,
        "output_dir": str(dst_city_dir.resolve()),
        "elapsed_ms": round((time.perf_counter() - t0) * 1000.0, 1),
        "rows_removed_total": sum(r["Удалено строк"] for r in file_stats),
        "kb_before": round(kb_before, 1),
        "kb_after": round(kb_after, 1),
        "saved_pct": round(100.0 * max(0.0, kb_before - kb_after) / kb_before, 2) if kb_before > 0 else 0.0,
        "pruned_transit_nodes_count": len(pruned_transit_nodes),
        "pruned_standby_pumps_count": len(pruned_standby_pumps),
        "pruned_zero_consumers_count": pruned_zero_consumers,
        "pruned_zero_demand_rows_count": pruned_zero_demand_rows,
        "stats_df": pd.DataFrame(file_stats),
    }

print("✅ Функции создания новых очищенных CSV готовы.")

✅ Функции создания новых очищенных CSV готовы.


## Клетка 7. Запуск создания новых очищенных файлов (`data_cleaned/<город>/*.csv`)
Выполняет очистку 4 таблиц для всех найденных городов и показывает точную сверку «До -> После» по каждому файлу.

In [8]:
CLEAN_RESULTS: list[dict[str, Any]] = []
single_city_mode = len(CITY_DIRS) == 1 and CITY_DIRS[0].resolve() == DATA_DIR.resolve()

for cdir in CITY_DIRS:
    if CLEAN_IN_PLACE:
        target_city_out = cdir
    elif single_city_mode:
        target_city_out = OUTPUT_CLEANED_DIR
    else:
        target_city_out = OUTPUT_CLEANED_DIR / cdir.name

    cres = clean_city_zero_rows(
        cdir,
        target_city_out,
        prune_node_head=PRUNE_NODE_HEAD_TOO,
    )
    CLEAN_RESULTS.append(cres)
    display(Markdown(
        f"### ✅ Созданы очищенные файлы для города `{cres['city_name']}` ({cres['elapsed_ms']} мс)\n"
        f"- **Папка новых файлов:** `{cres['output_dir']}`\n"
        f"- **Удалено нулевых строк всего:** `{cres['rows_removed_total']}` | "
        f"**Объём CSV:** `{cres['kb_before']} КБ -> {cres['kb_after']} КБ` (`-{cres['saved_pct']}%`)"
    ))
    display(cres["stats_df"])

### ✅ Созданы очищенные файлы для города `Алматы` (60178.2 мс)
- **Папка новых файлов:** `D:\Zhenechka\code\EPANET_AI\data_cleaned\Алматы`
- **Удалено нулевых строк всего:** `193768` | **Объём CSV:** `1673233.0 КБ -> 1331191.7 КБ` (`-20.44%`)

,Город,Таблица,Действие,Строк ДО,Строк ПОСЛЕ,Удалено строк,КБ ДО,КБ ПОСЛЕ
0,Алматы,Узел.csv,Очищена (без Требуемое давление),98006,0,98006,1824.0,0.1
1,Алматы,График потребления.csv,Скопирована без изменений,170,170,0,2.3,2.3
2,Алматы,Источник.csv,Скопирована без изменений,384,384,0,6.1,6.1
3,Алматы,Клапан.csv,Скопирована без изменений,254,254,0,8.5,8.5
4,Алматы,Кривая QH.csv,Скопирована без изменений,525,525,0,9.3,9.3
5,Алматы,Напор Узел.csv,Скопирована без изменений,98006,98006,0,974954.1,974954.1
6,Алматы,Насос.csv,Скопирована без изменений,701,701,0,25.4,25.4
7,Алматы,Потребители Узел.csv,Очищена (Потребление == 0),98006,50470,47536,1988.0,1330.6
8,Алматы,Потребление Узел.csv,Очищена от 0-строк,98006,49780,48226,690154.3,350594.3
9,Алматы,Труба.csv,Скопирована без изменений,102990,102990,0,4261.0,4261.0


### ✅ Созданы очищенные файлы для города `куляб` (1853.5 мс)
- **Папка новых файлов:** `D:\Zhenechka\code\EPANET_AI\data_cleaned\куляб`
- **Удалено нулевых строк всего:** `2566` | **Объём CSV:** `31757.2 КБ -> 30760.2 КБ` (`-3.14%`)

,Город,Таблица,Действие,Строк ДО,Строк ПОСЛЕ,Удалено строк,КБ ДО,КБ ПОСЛЕ
0,куляб,Узел.csv,Очищена (без Требуемое давление),1218,0,1218,22.4,0.1
1,куляб,График потребления.csv,Скопирована без изменений,169,169,0,2.3,2.3
2,куляб,Источник.csv,Скопирована без изменений,31,31,0,0.5,0.5
3,куляб,Клапан.csv,Скопирована без изменений,7,7,0,0.3,0.3
4,куляб,Кривая QH.csv,Скопирована без изменений,29,29,0,0.5,0.5
5,куляб,Напор Узел.csv,Скопирована без изменений,1218,1218,0,12007.8,12007.8
6,куляб,Насос.csv,Скопирована без изменений,49,49,0,1.7,1.7
7,куляб,Поток Клапан.csv,Скопирована без изменений,7,7,0,68.9,68.9
8,куляб,Поток Насос.csv,Очищена от 0-строк,49,33,16,399.8,287.2
9,куляб,Поток Труба.csv,Скопирована без изменений,1270,1270,0,10184.7,10184.7


### ✅ Созданы очищенные файлы для города `модель семей` (13278.8 мс)
- **Папка новых файлов:** `D:\Zhenechka\code\EPANET_AI\data_cleaned\модель семей`
- **Удалено нулевых строк всего:** `42258` | **Объём CSV:** `515728.8 КБ -> 512768.6 КБ` (`-0.57%`)

,Город,Таблица,Действие,Строк ДО,Строк ПОСЛЕ,Удалено строк,КБ ДО,КБ ПОСЛЕ
0,модель семей,Узел.csv,Очищена (без Требуемое давление),21004,0,21004,371.7,0.1
1,модель семей,График потребления.csv,Скопирована без изменений,74,74,0,0.9,0.9
2,модель семей,Источник.csv,Скопирована без изменений,54,54,0,0.9,0.9
3,модель семей,Клапан.csv,Скопирована без изменений,0,0,0,0.1,0.1
4,модель семей,Кривая QH.csv,Скопирована без изменений,23,23,0,0.4,0.4
5,модель семей,Напор Узел.csv,Скопирована без изменений,21004,21004,0,206970.9,206970.9
6,модель семей,Насос.csv,Скопирована без изменений,127,127,0,4.7,4.7
7,модель семей,Поток Клапан.csv,Скопирована без изменений,0,0,0,7.4,7.4
8,модель семей,Поток Насос.csv,Очищена от 0-строк,127,92,35,974.4,727.9
9,модель семей,Поток Труба.csv,Скопирована без изменений,21361,21361,0,156817.8,156817.8


... (2 outputs left out)

### ✅ Созданы очищенные файлы для города `тенино` (662.6 мс)
- **Папка новых файлов:** `D:\Zhenechka\code\EPANET_AI\data_cleaned\тенино`
- **Удалено нулевых строк всего:** `971` | **Объём CSV:** `13699.8 КБ -> 11335.0 КБ` (`-17.26%`)

,Город,Таблица,Действие,Строк ДО,Строк ПОСЛЕ,Удалено строк,КБ ДО,КБ ПОСЛЕ
0,тенино,Узел.csv,Очищена (без Требуемое давление),530,214,316,7.7,3.4
1,тенино,График потребления.csv,Скопирована без изменений,168,168,0,2.0,2.0
2,тенино,Источник.csv,Скопирована без изменений,1,1,0,0.0,0.0
3,тенино,Клапан.csv,Скопирована без изменений,3,3,0,0.1,0.1
4,тенино,Кривая QH.csv,Скопирована без изменений,126,126,0,1.6,1.6
5,тенино,Напор Узел.csv,Скопирована без изменений,530,530,0,5227.7,5227.7
6,тенино,Насос.csv,Скопирована без изменений,21,21,0,0.5,0.5
7,тенино,Поток Клапан.csv,Скопирована без изменений,3,3,0,33.8,33.8
8,тенино,Поток Насос.csv,Очищена от 0-строк,21,6,15,161.7,56.2
9,тенино,Поток Труба.csv,Скопирована без изменений,561,561,0,4496.7,4496.7


### ✅ Созданы очищенные файлы для города `хорезмская` (6050.5 мс)
- **Папка новых файлов:** `D:\Zhenechka\code\EPANET_AI\data_cleaned\хорезмская`
- **Удалено нулевых строк всего:** `21323` | **Объём CSV:** `248958.4 КБ -> 232817.7 КБ` (`-6.48%`)

,Город,Таблица,Действие,Строк ДО,Строк ПОСЛЕ,Удалено строк,КБ ДО,КБ ПОСЛЕ
0,хорезмская,Узел.csv,Очищена (без Требуемое давление),9878,0,9878,238.1,0.1
1,хорезмская,График потребления.csv,Скопирована без изменений,1994,1994,0,23.7,23.7
2,хорезмская,Источник.csv,Скопирована без изменений,15,15,0,0.3,0.3
3,хорезмская,Клапан.csv,Скопирована без изменений,8,8,0,0.4,0.4
4,хорезмская,Кривая QH.csv,Скопирована без изменений,164,164,0,2.7,2.7
5,хорезмская,Напор Узел.csv,Скопирована без изменений,9878,9878,0,90963.7,90963.7
6,хорезмская,Насос.csv,Скопирована без изменений,534,534,0,19.4,19.4
7,хорезмская,Поток Клапан.csv,Скопирована без изменений,8,8,0,83.4,83.4
8,хорезмская,Поток Насос.csv,Очищена от 0-строк,534,169,365,3855.6,1285.6
9,хорезмская,Поток Труба.csv,Скопирована без изменений,11194,11194,0,83135.5,83135.5


## Клетка 8. Расчёт и запись выводимых показателей (скорость, этажность, свободный напор, Re, мощность насосов)
Вычисляет все параметры, которые выводятся из комбинации таблиц города по вашим правилам и книге *Advanced Water Distribution Modeling and Management* (Walski et al., Haestad Press), и **записывает их в отдельные CSV-файлы внутри `data_cleaned/<город>/расчетные_параметры/`**:
- **Скорость в трубе:** и упрощённый индекс $|Q_{\max}| / D$, и точная скорость в круглом сечении $V = \frac{Q/3600}{\pi (D/1000)^2 / 4}$ м/с (Walski p. 40, Eq. 2.9) + полная матрица `Скорость Труба.csv` (`R0..R1439`);
- **Режим течения и время движения воды:** число Рейнольдса $\text{Re} = V D / \nu$ (Walski p. 42, Table 2.1), скоростной напор $V^2/(2g)$ (p. 43, Eq. 2.12), время пробега $t = L / V$ (мин, p. 67), объём воды в трубе $W = A \cdot L$ ($\text{м}^3$);
- **Этажность домов и свободный напор на узлах:** $n_{\text{эт}} = 1 + (H_{\text{треб}} - 10)/4$, свободный напор $H_{\text{своб}} = \text{HGL} - Z$ (м и бар, Walski p. 43, Eq. 2.12), запас/дефицит напора + полная матрица `Свободный напор Узел.csv`;
- **Рабочие точки и энергопотребление насосов:** моточасы, число пусков, интерполяция $H(Q)$ по `Кривая QH` (Walski p. 58) и гидравлическая мощность $N_{\text{гидр}} = \rho g Q H$ (кВт и кВт·ч/сут, Walski p. 295).

In [9]:
G_CONST = 9.80665            # ускорение свободного падения, м/с^2
NU_WATER_10C = 1.307e-6      # кинематическая вязкость воды при 10 °C, м^2/с (Walski Table 2.1)
BAR_PER_M = 0.0980665        # перевод м вод. ст. -> бар


def write_wide_derived_csv(out_path: Path, ids: list[str], r_cols: list[str], mat: np.ndarray, null_mask: np.ndarray, decimals: int = 3) -> None:
    """Быстро сохраняет рассчитанную широкую матрицу ID,R0..R1439 в CSV."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fmt = f"{{:.{decimals}f}}"
    with out_path.open("w", encoding="utf-8", newline="") as f:
        f.write(",".join(["ID"] + r_cols) + "\n")
        for i, oid in enumerate(ids):
            row_v = mat[i]
            row_n = null_mask[i]
            if not np.any(row_n) and not np.any(np.isnan(row_v)):
                f.write(oid + "," + ",".join(np.char.mod(f"%.{decimals}f", row_v).tolist()) + "\n")
            else:
                cells = ["" if (row_n[j] or math.isnan(float(row_v[j]))) else fmt.format(float(row_v[j])) for j in range(len(r_cols))]
                f.write(oid + "," + ",".join(cells) + "\n")


def read_static_csv_rows(fpath: Path) -> tuple[list[str], list[list[str]]]:
    enc, _, delim = detect_encoding_and_delimiter(fpath)
    txt = fpath.read_text(encoding=enc, errors="replace")
    rows = [r for r in csv.reader(io.StringIO(txt), delimiter=delim) if any(c.strip() for c in r)]
    return ([h.strip() for h in rows[0]], rows[1:]) if rows else ([], [])


def compute_derived_tables_for_city(city_folder: Path, save_wide: bool = True) -> dict[str, Any]:
    """Рассчитывает все взаимно выводимые показатели для одного города и сохраняет их в папку расчетные_параметры/."""
    derived_dir = city_folder / "расчетные_параметры"
    derived_dir.mkdir(parents=True, exist_ok=True)
    city_name = unicodedata.normalize("NFC", city_folder.name)

    by_canon: dict[str, Path] = {}
    for p in city_folder.iterdir():
        if p.is_file() and p.suffix.lower() == ".csv":
            sp = CITY_TABLE_SPECS.get(normalize_stem(p.name))
            if sp:
                by_canon[sp["canonical"]] = p

    created_files: list[dict[str, Any]] = []

    # Загружаем матрицу Напор Узел (HGL) для расчёта свободных напоров и потерь по трубам
    head_dict: dict[str, tuple[np.ndarray, np.ndarray]] = {}
    head_r_cols: list[str] = []
    if "wide/node_head.csv" in by_canon:
        h_ids, head_r_cols, h_mat, h_null, _, _ = load_wide_matrix_fast(by_canon["wide/node_head.csv"])
        for i, nid in enumerate(h_ids):
            if nid:
                head_dict[nid] = (h_mat[i], h_null[i])

    # Композитная базовая нагрузка из Потребители Узел (Walski p. 152)
    base_q_map: dict[str, float] = {}
    sched_map: dict[str, list[str]] = {}
    if "demands.csv" in by_canon:
        dh, drows = read_static_csv_rows(by_canon["demands.csv"])
        qi = next((i for i, h in enumerate(dh) if "потреблен" in h.lower()), 1)
        si = next((i for i, h in enumerate(dh) if "график" in h.lower()), 2)
        for r in drows:
            if not r:
                continue
            nid = r[0].strip()
            k, qv = parse_cell_num(r[qi]) if qi < len(r) else ("null", 0.0)
            if nid and k == "num" and qv > 0:
                base_q_map[nid] = base_q_map.get(nid, 0.0) + qv
            if nid and si < len(r) and r[si].strip() not in NULL_STRINGS:
                sc = r[si].strip()
                sched_map.setdefault(nid, [])
                if sc not in sched_map[nid]:
                    sched_map[nid].append(sc)

    # Фактический суточный объём из Потребление Узел (м3/сут)
    dem_stats_map: dict[str, tuple[float, float]] = {}
    if "wide/node_demand.csv" in by_canon:
        wd_ids, _, wd_mat, wd_null, _, _ = load_wide_matrix_fast(by_canon["wide/node_demand.csv"])
        dt_h = 24.0 / max(1, wd_mat.shape[1])
        for i, nid in enumerate(wd_ids):
            val_m = (~wd_null[i]) & (~np.isnan(wd_mat[i]))
            if np.any(val_m):
                arr = wd_mat[i][val_m]
                dem_stats_map[nid] = (float(np.sum(arr) * dt_h), float(np.max(arr)))

    # -------------------------------------------------------------------------
    # 1. УЗЛЫ: Этажность дома + Свободный напор (HGL - Z) + Запас / Дефицит
    # -------------------------------------------------------------------------
    nodes_df = pd.DataFrame()
    deficit_nodes = 0
    if "nodes.csv" in by_canon:
        nh, nrows = read_static_csv_rows(by_canon["nodes.csv"])
        zi = next((i for i, h in enumerate(nh) if "высот" in h.lower() or "отмет" in h.lower()), 1)
        ri = next((i for i, h in enumerate(nh) if "давлен" in h.lower()), 2)

        node_recs = []
        fp_ids, fp_rows, fp_nulls = [], [], []
        for r in nrows:
            if not r:
                continue
            nid = r[0].strip()
            kz, z_val = parse_cell_num(r[zi]) if zi < len(r) else ("null", 0.0)
            kr, h_req = parse_cell_num(r[ri]) if ri < len(r) else ("null", 0.0)
            z_m = z_val if kz == "num" else None
            req_m = h_req if kr == "num" else None

            if req_m is None or req_m <= 0.0:
                fl_exact, fl_int, btype = 0.0, 0, "Транзитный узел"
            elif req_m < 10.0:
                fl_exact, fl_int, btype = round(max(0.5, 1.0 + (req_m - 10.0) / 4.0), 2), 1, f"Ниже нормы 1 эт. ({req_m:g} м)"
            else:
                fl_exact = round(1.0 + (req_m - 10.0) / 4.0, 2)
                fl_int = max(1, int(round(fl_exact)))
                btype = "1 этаж (ИЖС)" if fl_int == 1 else (f"{fl_int} эт. (среднеэтажная)" if fl_int <= 5 else f"{fl_int} эт. (многоэтажная)")

            hgl_min = hgl_max = fp_min = fp_mean = fp_max = p_bar = margin = None
            def_mins = 0
            p_status = "Нет ряда Напор Узел"

            if nid in head_dict and z_m is not None:
                h_arr, h_nm = head_dict[nid]
                ok = (~h_nm) & (~np.isnan(h_arr))
                f_arr = h_arr - z_m
                if save_wide and req_m is not None and req_m > 0:
                    fp_ids.append(nid)
                    fp_rows.append(f_arr)
                    fp_nulls.append(h_nm | np.isnan(h_arr))
                if np.any(ok):
                    vh, vf = h_arr[ok], f_arr[ok]
                    hgl_min, hgl_max = round(float(np.min(vh)), 3), round(float(np.max(vh)), 3)
                    fp_min, fp_mean, fp_max = round(float(np.min(vf)), 3), round(float(np.mean(vf)), 3), round(float(np.max(vf)), 3)
                    p_bar = round(fp_max * BAR_PER_M, 3)
                    if req_m is not None and req_m > 0:
                        margin = round(fp_min - req_m, 3)
                        def_mins = int(round(float(np.sum(vf < req_m)) * (1440.0 / max(1, len(h_arr)))))
                        if margin < 0:
                            deficit_nodes += 1
                            p_status = f"ДЕФИЦИТ ({margin:+.2f} м)"
                        elif fp_max > 60.0:
                            p_status = f"ИЗБЫТОК > 60 м ({fp_max:.1f} м)"
                        else:
                            p_status = "Обеспечен (норма)"

            w_day, q_pk = dem_stats_map.get(nid, (0.0, 0.0))
            node_recs.append({
                "ID": nid,
                "Высота_Z_м": z_m,
                "Требуемое_давление_м": req_m,
                "Этажность_дома": fl_int,
                "Этажность_точная": fl_exact,
                "Тип_застройки": btype,
                "Базовый_расход_сумма_м3_ч": round(base_q_map.get(nid, 0.0), 3),
                "Пиковый_расход_м3_ч": round(q_pk, 3),
                "Суточный_объем_м3_сут": round(w_day, 2),
                "Графики": ",".join(sched_map.get(nid, [])),
                "HGL_мин_м": hgl_min,
                "HGL_макс_м": hgl_max,
                "Свободный_напор_мин_м": fp_min,
                "Свободный_напор_сред_м": fp_mean,
                "Свободный_напор_макс_м": fp_max,
                "Давление_макс_бар": p_bar,
                "Запас_напора_мин_м": margin,
                "Минут_дефицита_в_сутки": def_mins,
                "Статус_обеспеченности": p_status,
            })

        nodes_df = pd.DataFrame(node_recs)
        f_nodes = derived_dir / "Узел_этажность_и_свободный_напор.csv"
        nodes_df.to_csv(f_nodes, index=False, encoding="utf-8-sig")
        created_files.append({"Новый расчётный файл": f_nodes.name, "Строк": len(nodes_df), "Размер (КБ)": round(f_nodes.stat().st_size / 1024.0, 1), "Что содержит": "Этажность домов, свободный напор (HGL-Z), бар, дефициты и суточный объём"})

        if save_wide and fp_ids and head_r_cols:
            f_fp_wide = derived_dir / "Свободный напор Узел.csv"
            write_wide_derived_csv(f_fp_wide, fp_ids, head_r_cols, np.vstack(fp_rows), np.vstack(fp_nulls), decimals=2)
            created_files.append({"Новый расчётный файл": f_fp_wide.name, "Строк": len(fp_ids), "Размер (КБ)": round(f_fp_wide.stat().st_size / 1024.0, 1), "Что содержит": "Матрица свободного напора HGL - Z (м) на всех шагах R0..R1439"})

    # -------------------------------------------------------------------------
    # 2. ТРУБЫ: Скорость V=4Q/(pi*D^2), индекс Q/D, Рейнольдс Re, Время пробега L/V
    # -------------------------------------------------------------------------
    pipes_df = pd.DataFrame()
    if "pipes.csv" in by_canon:
        ph, prows = read_static_csv_rows(by_canon["pipes.csv"])
        si = next((i for i, h in enumerate(ph) if "начал" in h.lower()), 1)
        ei = next((i for i, h in enumerate(ph) if "конец" in h.lower()), 2)
        di = next((i for i, h in enumerate(ph) if "диаметр" in h.lower()), 3)
        li = next((i for i, h in enumerate(ph) if "длин" in h.lower()), 4)

        pf_dict: dict[str, tuple[np.ndarray, np.ndarray]] = {}
        pf_r_cols: list[str] = []
        if "wide/pipe_flow.csv" in by_canon:
            pf_ids, pf_r_cols, pf_mat, pf_null, _, _ = load_wide_matrix_fast(by_canon["wide/pipe_flow.csv"])
            for i, pid in enumerate(pf_ids):
                if pid:
                    pf_dict[pid] = (pf_mat[i], pf_null[i])

        pipe_recs = []
        v_ids, v_rows, v_nulls = [], [], []
        for r in prows:
            if not r:
                continue
            pid = r[0].strip()
            n1 = r[si].strip() if si < len(r) else ""
            n2 = r[ei].strip() if ei < len(r) else ""
            kd, d_val = parse_cell_num(r[di]) if di < len(r) else ("null", 0.0)
            kl, l_val = parse_cell_num(r[li]) if li < len(r) else ("null", 0.0)
            d_mm = d_val if (kd == "num" and d_val > 0) else None
            l_m = l_val if (kl == "num" and l_val > 0) else None
            d_m = (d_mm / 1000.0) if d_mm else None
            area_m2 = (math.pi * d_m * d_m / 4.0) if d_m else None
            vol_m3 = round(area_m2 * l_m, 3) if (area_m2 and l_m) else None

            q_mean = q_max = q_over_d = v_min = v_mean = v_max = hv_m = re_max = t_run_min = None
            rev_cnt = 0
            re_regime = walski_eval = "Нет ряда потока"

            if pid in pf_dict and area_m2 and d_m and d_mm:
                q_arr, q_nm = pf_dict[pid]
                ok = (~q_nm) & (~np.isnan(q_arr))
                # Точная скорость по книге Walski p. 40, Eq. 2.9: V = (Q / 3600) / (pi * D^2 / 4)
                v_arr = (q_arr / 3600.0) / area_m2
                if save_wide and pf_r_cols:
                    v_ids.append(pid)
                    v_rows.append(v_arr)
                    v_nulls.append(q_nm | np.isnan(q_arr))
                if np.any(ok):
                    vq = q_arr[ok]
                    vv = np.abs(v_arr[ok])
                    q_mean = round(float(np.mean(np.abs(vq))), 3)
                    q_max = round(float(np.max(np.abs(vq))), 3)
                    q_over_d = round(q_max / d_mm, 4)  # ваш упрощённый показатель Поток / Диаметр
                    v_min = round(float(np.min(vv)), 4)
                    v_mean = round(float(np.mean(vv)), 4)
                    v_max = round(float(np.max(vv)), 4)
                    hv_m = round((v_max * v_max) / (2.0 * G_CONST), 4)
                    re_max = int(round((v_max * d_m) / NU_WATER_10C))
                    re_regime = (
                        "Застой (V=0)" if v_max <= 1e-9 else
                        ("Ламинарный (<2000)" if re_max < 2000 else ("Переходный (2000..4000)" if re_max <= 4000 else "Турбулентный (>4000)"))
                    )
                    walski_eval = (
                        "Нет движения" if v_max <= 1e-6 else
                        ("Малая <0.3 м/с (риск застоя)" if v_max < 0.3 else
                         ("Норма сети 0.3..1.5 м/с" if v_max <= 1.5 else
                          ("Водовод 1.5..3.0 м/с (Walski p.292)" if v_max <= 3.0 else "Высокая >3.0 м/с (риск гидроудара!)")))
                    )
                    rev_cnt = int(np.sum(vq < 0.0))
                    if l_m and v_mean > 1e-6:
                        t_run_min = round(l_m / (v_mean * 60.0), 2)

            dh_max = slope_m_km = None
            if n1 and n1 != n2 and n1 in head_dict and n2 in head_dict:
                h1, nm1 = head_dict[n1]
                h2, nm2 = head_dict[n2]
                both = (~nm1) & (~nm2) & (~np.isnan(h1)) & (~np.isnan(h2))
                if np.any(both):
                    dh_max = round(float(np.max(np.abs(h1[both] - h2[both]))), 3)
                    if l_m:
                        slope_m_km = round((dh_max / l_m) * 1000.0, 3)

            pipe_recs.append({
                "ID": pid,
                "Начало": n1,
                "Конец": n2,
                "Диаметр_мм": d_mm,
                "Длина_м": l_m,
                "Площадь_сечения_м2": round(area_m2, 6) if area_m2 else None,
                "Объем_трубы_м3": vol_m3,
                "Расход_сред_м3_ч": q_mean,
                "Расход_макс_м3_ч": q_max,
                "Индекс_Поток_делить_Диаметр": q_over_d,
                "Скорость_мин_м_с": v_min,
                "Скорость_сред_м_с": v_mean,
                "Скорость_макс_м_с": v_max,
                "Скоростной_напор_V2_2g_м": hv_m,
                "Число_Рейнольдса_Re": re_max,
                "Режим_течения_Walski": re_regime,
                "Время_пробега_воды_мин": t_run_min,
                "Шагов_обратного_потока": rev_cnt,
                "Потери_напора_макс_м": dh_max,
                "Гидравлический_уклон_м_км": slope_m_km,
                "Оценка_скорости_Walski": walski_eval,
            })

        pipes_df = pd.DataFrame(pipe_recs)
        f_pipes = derived_dir / "Труба_скорости_и_режимы_Walski.csv"
        pipes_df.to_csv(f_pipes, index=False, encoding="utf-8-sig")
        created_files.append({"Новый расчётный файл": f_pipes.name, "Строк": len(pipes_df), "Размер (КБ)": round(f_pipes.stat().st_size / 1024.0, 1), "Что содержит": "Скорость V=4Q/(pi*D^2), индекс Q/D, Re, время пробега L/V, объём воды, уклон"})

        if save_wide and v_ids and pf_r_cols:
            f_vel_wide = derived_dir / "Скорость Труба.csv"
            write_wide_derived_csv(f_vel_wide, v_ids, pf_r_cols, np.vstack(v_rows), np.vstack(v_nulls), decimals=4)
            created_files.append({"Новый расчётный файл": f_vel_wide.name, "Строк": len(v_ids), "Размер (КБ)": round(f_vel_wide.stat().st_size / 1024.0, 1), "Что содержит": "Знаковая матрица скоростей потока V (м/с) на всех шагах R0..R1439"})

    # -------------------------------------------------------------------------
    # 3. НАСОСЫ: Моточасы, пуски, интерполяция Q-H и гидравлическая мощность (кВт)
    # -------------------------------------------------------------------------
    pumps_df = pd.DataFrame()
    if "pumps.csv" in by_canon:
        pmh, pmrows = read_static_csv_rows(by_canon["pumps.csv"])
        ci = next((i for i, h in enumerate(pmh) if "крив" in h.lower()), 3)

        qh_curves: dict[str, list[tuple[float, float]]] = {}
        if "pump_curves.csv" in by_canon:
            _, qhrows = read_static_csv_rows(by_canon["pump_curves.csv"])
            for r in qhrows:
                if len(r) >= 3:
                    cid = r[0].strip()
                    kx, xv = parse_cell_num(r[1])
                    ky, yv = parse_cell_num(r[2])
                    if cid and kx == "num" and ky == "num":
                        qh_curves.setdefault(cid, []).append((xv, yv))
            for cid in qh_curves:
                qh_curves[cid].sort(key=lambda t: t[0])

        pmf_dict: dict[str, tuple[np.ndarray, np.ndarray]] = {}
        if "wide/pump_flow.csv" in by_canon:
            pm_ids, _, pm_mat, pm_null, _, _ = load_wide_matrix_fast(by_canon["wide/pump_flow.csv"])
            for i, pmid in enumerate(pm_ids):
                if pmid:
                    pmf_dict[pmid] = (pm_mat[i], pm_null[i])

        pump_recs = []
        for r in pmrows:
            if not r:
                continue
            pmid = r[0].strip()
            cid = r[ci].strip() if ci < len(r) else ""
            pts = qh_curves.get(cid, [])
            xs = np.array([p[0] for p in pts], dtype=np.float64) if pts else np.array([])
            ys = np.array([p[1] for p in pts], dtype=np.float64) if pts else np.array([])
            h0 = float(ys[0]) if pts else None

            if pmid not in pmf_dict:
                pump_recs.append({
                    "ID": pmid, "Кривая_QH": cid, "Режим_работы": "В резерве (удалён из Поток Насос как 0.00)",
                    "Моточасов_в_сутки": 0.0, "Число_пусков": 0, "Подача_сред_м3_ч": 0.0, "Подача_макс_м3_ч": 0.0,
                    "Суточная_подача_м3_сут": 0.0, "Напор_H0_м": h0, "Расчетный_напор_TDH_м": None,
                    "Гидравл_мощность_сред_кВт": 0.0, "Гидравл_энергия_кВтч_сут": 0.0,
                })
                continue

            q_arr, q_nm = pmf_dict[pmid]
            ok = (~q_nm) & (~np.isnan(q_arr))
            dt_h = 24.0 / max(1, len(q_arr))
            vq = q_arr[ok]
            act = vq > 1e-6
            if not np.any(act):
                pump_recs.append({
                    "ID": pmid, "Кривая_QH": cid, "Режим_работы": "В резерве (выключен все сутки)",
                    "Моточасов_в_сутки": 0.0, "Число_пусков": 0, "Подача_сред_м3_ч": 0.0, "Подача_макс_м3_ч": 0.0,
                    "Суточная_подача_м3_сут": 0.0, "Напор_H0_м": h0, "Расчетный_напор_TDH_м": None,
                    "Гидравл_мощность_сред_кВт": 0.0, "Гидравл_энергия_кВтч_сут": 0.0,
                })
                continue

            qa = vq[act]
            hours_on = round(float(np.sum(act)) * dt_h, 2)
            on_i = act.astype(np.int8)
            starts = int(np.sum((on_i[1:] == 1) & (on_i[:-1] == 0))) + (1 if on_i[0] == 1 and not np.all(act) else 0)
            tdh_m = p_kw = e_kwh = None
            if len(pts) >= 2:
                hi = np.interp(qa, xs, ys)
                pk = (G_CONST * qa * hi) / 3600.0
                tdh_m, p_kw, e_kwh = round(float(np.mean(hi)), 2), round(float(np.mean(pk)), 2), round(float(np.sum(pk)) * dt_h, 2)
            elif len(pts) == 1:
                tdh_m = float(ys[0])
                p_kw = round((G_CONST * float(np.mean(qa)) * tdh_m) / 3600.0, 2)
                e_kwh = round(p_kw * hours_on, 2)

            pump_recs.append({
                "ID": pmid,
                "Кривая_QH": cid,
                "Режим_работы": "Постоянно (24 ч)" if np.all(act) else f"Циклический ({hours_on:g} ч/сут)",
                "Моточасов_в_сутки": hours_on,
                "Число_пусков": starts,
                "Подача_сред_м3_ч": round(float(np.mean(qa)), 2),
                "Подача_макс_м3_ч": round(float(np.max(qa)), 2),
                "Суточная_подача_м3_сут": round(float(np.sum(qa)) * dt_h, 2),
                "Напор_H0_м": h0,
                "Расчетный_напор_TDH_м": tdh_m,
                "Гидравл_мощность_сред_кВт": p_kw,
                "Гидравл_энергия_кВтч_сут": e_kwh,
            })

        pumps_df = pd.DataFrame(pump_recs)
        f_pumps = derived_dir / "Насос_рабочие_точки_и_мощность.csv"
        pumps_df.to_csv(f_pumps, index=False, encoding="utf-8-sig")
        created_files.append({"Новый расчётный файл": f_pumps.name, "Строк": len(pumps_df), "Размер (КБ)": round(f_pumps.stat().st_size / 1024.0, 1), "Что содержит": "Моточасы, пуски, напор TDH по кривой QH, мощность (кВт) и энергия (кВт*ч/сут)"})

    # -------------------------------------------------------------------------
    # 4. ГРАФИКИ ПОТРЕБЛЕНИЯ: Пиковые коэффициенты K_min, K_сред, K_max
    # -------------------------------------------------------------------------
    if "irregularity_curves.csv" in by_canon:
        _, srows = read_static_csv_rows(by_canon["irregularity_curves.csv"])
        s_dict: dict[str, list[float]] = {}
        for r in srows:
            if len(r) >= 2:
                cid = r[0].strip()
                k, kv = parse_cell_num(r[1])
                if cid and k == "num":
                    s_dict.setdefault(cid, []).append(kv)
        s_recs = []
        for cid, vals in sorted(s_dict.items()):
            arr = np.array(vals, dtype=np.float64)
            s_recs.append({
                "ID": cid,
                "Строк_часов": len(vals),
                "K_мин": round(float(np.min(arr)), 5),
                "K_сред": round(float(np.mean(arr)), 5),
                "K_макс": round(float(np.max(arr)), 5),
                "Час_максимума": f"{int(np.argmax(arr)):02d}:00",
                "Коэф_неравномерности_Kмакс_Kмин": round(float(np.max(arr) / np.min(arr)), 3) if np.min(arr) > 1e-6 else None,
            })
        f_sched = derived_dir / "График_потребления_анализ_24ч.csv"
        pd.DataFrame(s_recs).to_csv(f_sched, index=False, encoding="utf-8-sig")
        created_files.append({"Новый расчётный файл": f_sched.name, "Строк": len(s_recs), "Размер (КБ)": round(f_sched.stat().st_size / 1024.0, 1), "Что содержит": "Коэффициенты часовой неравномерности K_min, K_сред, K_max и час пика"})

    # Паспорт формул
    (derived_dir / "derived_formulas_manifest.json").write_text(
        json.dumps({
            "city": city_name,
            "pipe_velocity_formula": "V [м/с] = 4 * (Q / 3600) / (pi * (D / 1000)^2) (Walski p. 40, Eq. 2.9)",
            "pipe_q_div_d_index": "|Q_max| / D_mm",
            "reynolds_number": "Re = V * D / nu, nu=1.307e-6 м2/с (Walski p. 42, Table 2.1)",
            "free_pressure_m": "H_free = HGL - Z (Walski p. 43, Eq. 2.12)",
            "storeys_formula": "n = 1 + (H_req - 10) / 4 (СП 31.13330.2021)",
            "pump_power_kw": "N = 9.80665 * Q * H(Q) / 3600 (Walski p. 58, p. 295)",
        }, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

    return {
        "city_name": city_name,
        "derived_dir": str(derived_dir.resolve()),
        "files_df": pd.DataFrame(created_files),
        "nodes_preview": nodes_df.head(5),
        "pipes_preview": pipes_df.head(5),
        "pumps_preview": pumps_df.head(5),
        "deficit_nodes": deficit_nodes,
    }


DERIVED_RESULTS = []
if COMPUTE_DERIVED_WALSKI_TABLES:
    for cres in CLEAN_RESULTS:
        dres = compute_derived_tables_for_city(
            Path(cres["output_dir"]),
            save_wide=SAVE_WIDE_VELOCITY_AND_PRESSURE,
        )
        DERIVED_RESULTS.append(dres)
        display(Markdown(
            f"### 📐 Записаны расчётные показатели для города `{dres['city_name']}`\n"
            f"- **Папка новых расчётных файлов:** `{dres['derived_dir']}`"
        ))
        display(dres["files_df"])
        if not dres["nodes_preview"].empty:
            display(Markdown("**Пример рассчитанной этажности и свободного напора (`Узел_этажность_и_свободный_напор.csv`, первые 5 строк):**"))
            display(dres["nodes_preview"])
        if not dres["pipes_preview"].empty:
            display(Markdown("**Пример рассчитанных скоростей, индекса Q/D, Re и времени пробега (`Труба_скорости_и_режимы_Walski.csv`, первые 5 строк):**"))
            display(dres["pipes_preview"])
        if not dres["pumps_preview"].empty:
            display(Markdown("**Пример рабочих точек и мощности насосов (`Насос_рабочие_точки_и_мощность.csv`):**"))
            display(dres["pumps_preview"])

### 📐 Записаны расчётные показатели для города `Алматы`
- **Папка новых расчётных файлов:** `D:\Zhenechka\code\EPANET_AI\data_cleaned\Алматы\расчетные_параметры`

,Новый расчётный файл,Строк,Размер (КБ),Что содержит
0,Узел_этажность_и_свободный_напор.csv,0,0.0,"Этажность домов, свободный напор (HGL-Z), бар, дефициты и суточный объём"
1,Труба_скорости_и_режимы_Walski.csv,102990,12530.4,"Скорость V=4Q/(pi*D^2), индекс Q/D, Re, время пробега L/V, объём воды, уклон"
2,Насос_рабочие_точки_и_мощность.csv,701,84.8,"Моточасы, пуски, напор TDH по кривой QH, мощность (кВт) и энергия (кВт*ч/сут)"
3,График_потребления_анализ_24ч.csv,9,0.5,"Коэффициенты часовой неравномерности K_min, K_сред, K_max и час пика"


**Пример рассчитанных скоростей, индекса Q/D, Re и времени пробега (`Труба_скорости_и_режимы_Walski.csv`, первые 5 строк):**

,ID,Начало,Конец,Диаметр_мм,Длина_м,Площадь_сечения_м2,Объем_трубы_м3,Расход_сред_м3_ч,Расход_макс_м3_ч,Индекс_Поток_делить_Диаметр,Скорость_мин_м_с,Скорость_сред_м_с,Скорость_макс_м_с,Скоростной_напор_V2_2g_м,Число_Рейнольдса_Re,Режим_течения_Walski,Время_пробега_воды_мин,Шагов_обратного_потока,Потери_напора_макс_м,Гидравлический_уклон_м_км,Оценка_скорости_Walski
0,e3575368,c3575368,c3575368,150.0,80.0,0.017671,1.414,None,None,None,None,None,None,None,None,Нет ряда потока,None,0,None,None,Нет ряда потока
1,e3575369,c3575369,c3575369,1200.0,71.0,1.130973,80.299,None,None,None,None,None,None,None,None,Нет ряда потока,None,0,None,None,Нет ряда потока
2,e3575370,c3575370,c3575370,150.0,80.0,0.017671,1.414,None,None,None,None,None,None,None,None,Нет ряда потока,None,0,None,None,Нет ряда потока
3,e3575371,c3575371,c3575371,150.0,80.0,0.017671,1.414,None,None,None,None,None,None,None,None,Нет ряда потока,None,0,None,None,Нет ряда потока
4,e3575372,c3575372,c3575372,50.0,80.0,0.001963,0.157,None,None,None,None,None,None,None,None,Нет ряда потока,None,0,None,None,Нет ряда потока


**Пример рабочих точек и мощности насосов (`Насос_рабочие_точки_и_мощность.csv`):**

,ID,Кривая_QH,Режим_работы,Моточасов_в_сутки,Число_пусков,Подача_сред_м3_ч,Подача_макс_м3_ч,Суточная_подача_м3_сут,Напор_H0_м,Расчетный_напор_TDH_м,Гидравл_мощность_сред_кВт,Гидравл_энергия_кВтч_сут
0,p3678218,u3682003,В резерве (удалён из Поток Насос как 0.00),0.0,0,0.0,0.0,0.0,50.0,None,0.0,0.0
1,p3678219,u3682358,В резерве (удалён из Поток Насос как 0.00),0.0,0,0.0,0.0,0.0,105.0,None,0.0,0.0
2,p3678220,u3682358,В резерве (удалён из Поток Насос как 0.00),0.0,0,0.0,0.0,0.0,105.0,None,0.0,0.0
3,p3678221,u3476140,В резерве (удалён из Поток Насос как 0.00),0.0,0,0.0,0.0,0.0,90.0,None,0.0,0.0
4,p3678222,u3680999,В резерве (удалён из Поток Насос как 0.00),0.0,0,0.0,0.0,0.0,90.0,None,0.0,0.0


... (30 outputs left out)

**Пример рассчитанных скоростей, индекса Q/D, Re и времени пробега (`Труба_скорости_и_режимы_Walski.csv`, первые 5 строк):**

,ID,Начало,Конец,Диаметр_мм,Длина_м,Площадь_сечения_м2,Объем_трубы_м3,Расход_сред_м3_ч,Расход_макс_м3_ч,Индекс_Поток_делить_Диаметр,Скорость_мин_м_с,Скорость_сред_м_с,Скорость_макс_м_с,Скоростной_напор_V2_2g_м,Число_Рейнольдса_Re,Режим_течения_Walski,Время_пробега_воды_мин,Шагов_обратного_потока,Потери_напора_макс_м,Гидравлический_уклон_м_км,Оценка_скорости_Walski
0,e1024726,c1024726,c1024726,500.0,80.0,0.19635,15.708,57.708,181.91,0.3638,0.0,0.0816,0.2574,0.0034,98470,Турбулентный (>4000),16.34,0,None,None,Малая <0.3 м/с (риск застоя)
1,e1024728,c1024728,c1024728,500.0,80.0,0.19635,15.708,43.643,139.29,0.2786,0.0,0.0617,0.1971,0.0020,75402,Турбулентный (>4000),21.61,1111,None,None,Малая <0.3 м/с (риск застоя)
2,e1024730,c1024730,c1024730,500.0,80.0,0.19635,15.708,18.883,96.02,0.1920,0.0,0.0267,0.1358,0.0009,51951,Турбулентный (>4000),49.94,798,None,None,Малая <0.3 м/с (риск застоя)
3,e1024732,c1024732,c1024732,500.0,80.0,0.19635,15.708,52.617,223.81,0.4476,0.0,0.0744,0.3166,0.0051,121117,Турбулентный (>4000),17.92,751,None,None,Норма сети 0.3..1.5 м/с
4,e1024734,c1024734,c1024734,500.0,80.0,0.19635,15.708,52.617,223.81,0.4476,0.0,0.0744,0.3166,0.0051,121117,Турбулентный (>4000),17.92,751,None,None,Норма сети 0.3..1.5 м/с


**Пример рабочих точек и мощности насосов (`Насос_рабочие_точки_и_мощность.csv`):**

,ID,Кривая_QH,Режим_работы,Моточасов_в_сутки,Число_пусков,Подача_сред_м3_ч,Подача_макс_м3_ч,Суточная_подача_м3_сут,Напор_H0_м,Расчетный_напор_TDH_м,Гидравл_мощность_сред_кВт,Гидравл_энергия_кВтч_сут
0,p1024700,u1024823,Циклический (6 ч/сут),6.0,3,135.92,181.91,815.52,57.0,57.0,21.10,126.60
1,p1024703,u1024823,В резерве (удалён из Поток Насос как 0.00),0.0,0,0.00,0.00,0.00,57.0,NaN,0.00,0.00
2,p1024706,u1024823,Циклический (6 ч/сут),6.0,3,118.98,164.96,713.88,57.0,57.0,18.47,110.82
3,p1024709,u1024823,Циклический (6 ч/сут),6.0,3,135.93,181.92,815.58,57.0,57.0,21.11,126.66
4,p1024712,u1024830,В резерве (удалён из Поток Насос как 0.00),0.0,0,0.00,0.00,0.00,56.0,NaN,0.00,0.00


## Клетка 9. Создание компактных аналитических сводок (`wide_summary/*.csv` и `.parquet`)
По очищенным таблицам каждого города формируются компактные инженерные таблицы сводок (дескриптивные агрегаты `min`, `max`, `mean`, `range`, длительность нулевого и обратного потока `Q < 0`, а также 24 часовых среза `h00..h23` — **строго без пересчёта гидравлики в Python, Принципы 1–2**).

In [10]:
def build_wide_summaries_for_city(cleaned_city_dir: Path, summary_city_out: Path) -> list[dict[str, Any]]:
    """Создаёт компактные сводные файлы CSV (и Parquet при наличии pyarrow) по широким матрицам R0..R1439."""
    if not EXPORT_PARQUET_SUMMARIES or not cleaned_city_dir.exists():
        return []
    summary_city_out.mkdir(parents=True, exist_ok=True)
    created_summaries = []

    for fpath in sorted(cleaned_city_dir.iterdir()):
        if not fpath.is_file() or fpath.suffix.lower() != ".csv":
            continue
        spec = CITY_TABLE_SPECS.get(normalize_stem(fpath.name))
        if spec is None or not spec["is_wide"]:
            continue

        ids, r_cols, mat, null_mask, _, _ = load_wide_matrix_fast(fpath)
        n_rows, n_steps = mat.shape
        if n_rows == 0 or n_steps == 0:
            continue

        ieee_nan = (~null_mask) & np.isnan(mat)
        valid = (~null_mask) & (~ieee_nan)
        safe_pos = np.where(valid, mat, np.inf)
        safe_neg = np.where(valid, mat, -np.inf)
        safe_sum = np.where(valid, mat, 0.0)
        valid_cnt = np.sum(valid, axis=1)

        with np.errstate(invalid="ignore", divide="ignore"):
            v_min = np.where(valid_cnt > 0, np.min(safe_pos, axis=1), np.nan)
            v_max = np.where(valid_cnt > 0, np.max(safe_neg, axis=1), np.nan)
            v_mean = np.where(valid_cnt > 0, np.sum(safe_sum, axis=1) / np.maximum(valid_cnt, 1), np.nan)
            v_range = v_max - v_min

        zero_steps = np.sum(valid & (mat == 0.0), axis=1)
        neg_steps = np.sum(valid & (mat < 0.0), axis=1)
        null_steps = np.sum(null_mask, axis=1)

        # Подсчёт смен знака потока (реверсов направления)
        signs = np.where(valid & (np.abs(mat) > 1e-9), np.sign(mat), 0.0)
        reversals = np.zeros(n_rows, dtype=np.int64)
        for i in range(n_rows):
            nz = signs[i][signs[i] != 0.0]
            if nz.size >= 2:
                reversals[i] = int(np.count_nonzero(nz[1:] != nz[:-1]))

        summary_data: dict[str, Any] = {
            "id": ids,
            "source_file": [fpath.name] * n_rows,
            "valid_steps_count": valid_cnt,
            "null_steps_count": null_steps,
            "zero_steps_count": zero_steps,
            "negative_steps_count": neg_steps,
            "flow_reversals_count": reversals,
            "min_val": np.round(v_min, 4),
            "max_val": np.round(v_max, 4),
            "mean_val": np.round(v_mean, 4),
            "range_val": np.round(v_range, 4),
        }

        # Если 1440 минутных шагов — сохраняем 24 точных часовых узла R0, R60, ..., R1380
        if n_steps >= 1440:
            for h in range(24):
                summary_data[f"h{h:02d}"] = mat[:, h * 60]

        sum_df = pd.DataFrame(summary_data)
        safe_slug = spec["canonical"].replace("wide/", "").replace(".csv", "")
        csv_out = summary_city_out / f"summary_{safe_slug}.csv"
        sum_df.to_csv(csv_out, index=False, encoding="utf-8-sig")

        pq_size_kb = None
        if HAS_PYARROW:
            pq_out = summary_city_out / f"summary_{safe_slug}.parquet"
            pq.write_table(pa.Table.from_pandas(sum_df, preserve_index=False), pq_out, compression="zstd")
            pq_size_kb = round(pq_out.stat().st_size / 1024.0, 1)

        created_summaries.append({
            "Исходная матрица": fpath.name,
            "Сводный CSV": csv_out.name,
            "Строк": n_rows,
            "Исходный размер (КБ)": round(fpath.stat().st_size / 1024.0, 1),
            "Сводный CSV (КБ)": round(csv_out.stat().st_size / 1024.0, 1),
            "Сводный Parquet (КБ)": pq_size_kb if pq_size_kb is not None else "—",
        })
    return created_summaries

SUMMARY_TABLES_LOG = []
for cres in CLEAN_RESULTS:
    c_out = Path(cres["output_dir"])
    s_out = OUTPUT_SUMMARY_DIR / "summaries" / cres["city_name"]
    s_items = build_wide_summaries_for_city(c_out, s_out)
    if s_items:
        SUMMARY_TABLES_LOG.extend(s_items)
        display(Markdown(f"#### Компактные аналитические сводки `wide_summary` для `{cres['city_name']}`:"))
        display(pd.DataFrame(s_items))

#### Компактные аналитические сводки `wide_summary` для `Алматы`:

,Исходная матрица,Сводный CSV,Строк,Исходный размер (КБ),Сводный CSV (КБ),Сводный Parquet (КБ)
0,Напор Узел.csv,summary_node_head.csv,98006,974954.1,23232.2,6651.8
1,Потребление Узел.csv,summary_node_demand.csv,49780,350594.3,9688.4,1635.3


#### Компактные аналитические сводки `wide_summary` для `куляб`:

,Исходная матрица,Сводный CSV,Строк,Исходный размер (КБ),Сводный CSV (КБ),Сводный Parquet (КБ)
0,Напор Узел.csv,summary_node_head.csv,1218,12007.8,285.6,144.3
1,Поток Клапан.csv,summary_valve_flow.csv,7,68.9,1.7,20.9
2,Поток Насос.csv,summary_pump_flow.csv,33,287.2,7.0,24.5
3,Поток Труба.csv,summary_pipe_flow.csv,1270,10184.7,256.1,161.8
4,Потребление Узел.csv,summary_node_demand.csv,1098,8012.9,218.0,115.1


#### Компактные аналитические сводки `wide_summary` для `модель семей`:

,Исходная матрица,Сводный CSV,Строк,Исходный размер (КБ),Сводный CSV (КБ),Сводный Parquet (КБ)
0,Напор Узел.csv,summary_node_head.csv,21004,206970.9,4940.0,1024.4
1,Поток Насос.csv,summary_pump_flow.csv,92,727.9,18.4,34.3
2,Поток Труба.csv,summary_pipe_flow.csv,21361,156817.8,4061.5,1237.8
3,Потребление Узел.csv,summary_node_demand.csv,20712,145875.6,4045.2,506.1


... (2 outputs left out)

#### Компактные аналитические сводки `wide_summary` для `тенино`:

,Исходная матрица,Сводный CSV,Строк,Исходный размер (КБ),Сводный CSV (КБ),Сводный Parquet (КБ)
0,Напор Узел.csv,summary_node_head.csv,530,5227.7,122.4,50.4
1,Поток Клапан.csv,summary_valve_flow.csv,3,33.8,0.9,20.3
2,Поток Насос.csv,summary_pump_flow.csv,6,56.2,1.4,21.0
3,Поток Труба.csv,summary_pipe_flow.csv,561,4496.7,111.7,78.9
4,Потребление Узел.csv,summary_node_demand.csv,210,1490.1,40.2,39.1


#### Компактные аналитические сводки `wide_summary` для `хорезмская`:

,Исходная матрица,Сводный CSV,Строк,Исходный размер (КБ),Сводный CSV (КБ),Сводный Parquet (КБ)
0,Напор Узел.csv,summary_node_head.csv,9878,90963.7,2198.3,524.9
1,Поток Клапан.csv,summary_valve_flow.csv,8,83.4,2.1,21.2
2,Поток Насос.csv,summary_pump_flow.csv,169,1285.6,31.0,33.4
3,Поток Труба.csv,summary_pipe_flow.csv,11194,83135.5,2074.7,673.0
4,Потребление Узел.csv,summary_node_demand.csv,8007,56623.7,1504.1,281.7


## Клетка 10. Сохранение итоговых отчётов (`HTML`, `Markdown`, `JSON`) и визуальная сводка
Сохраняет автономный отчёт в `OUTPUT_SUMMARY_DIR/reports/` и выводит интерактивную сводную карточку прямо под клеткой.

In [11]:
REPORTS_DIR = OUTPUT_SUMMARY_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if CLEAN_RESULTS:
    total_removed = sum(c["rows_removed_total"] for c in CLEAN_RESULTS)
    total_kb_before = round(sum(c["kb_before"] for c in CLEAN_RESULTS), 1)
    total_kb_after = round(sum(c["kb_after"] for c in CLEAN_RESULTS), 1)
    total_saved_pct = round(100.0 * max(0.0, total_kb_before - total_kb_after) / total_kb_before, 2) if total_kb_before > 0 else 0.0

    summary_payload = {
        "project": "EPANET-AI",
        "cities_processed": len(CLEAN_RESULTS),
        "total_rows_removed": total_removed,
        "total_kb_before": total_kb_before,
        "total_kb_after": total_kb_after,
        "saved_pct": total_saved_pct,
        "cities": [
            {k: v for k, v in c.items() if k != "stats_df"}
            for c in CLEAN_RESULTS
        ],
    }

    json_report_path = REPORTS_DIR / "epanet_ai_zero_clean_report.json"
    json_report_path.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    city_rows_html = "".join(
        f"""<tr>
            <td style="padding:8px;border-bottom:1px solid #e2e8f0;"><b>{html.escape(c['city_name'])}</b></td>
            <td style="padding:8px;border-bottom:1px solid #e2e8f0;text-align:right;">{c['pruned_transit_nodes_count']}</td>
            <td style="padding:8px;border-bottom:1px solid #e2e8f0;text-align:right;">{c['pruned_standby_pumps_count']}</td>
            <td style="padding:8px;border-bottom:1px solid #e2e8f0;text-align:right;">{c['pruned_zero_consumers_count']}</td>
            <td style="padding:8px;border-bottom:1px solid #e2e8f0;text-align:right;">{c['pruned_zero_demand_rows_count']}</td>
            <td style="padding:8px;border-bottom:1px solid #e2e8f0;text-align:right;font-weight:700;color:#15803d;">{c['rows_removed_total']}</td>
            <td style="padding:8px;border-bottom:1px solid #e2e8f0;text-align:right;">{c['kb_before']} &rarr; {c['kb_after']} КБ (&minus;{c['saved_pct']}%)</td>
            <td style="padding:8px;border-bottom:1px solid #e2e8f0;font-size:12px;"><code>{html.escape(c['output_dir'])}</code></td>
        </tr>"""
        for c in CLEAN_RESULTS
    )

    report_card_html = f"""
    <div style="font-family:'Segoe UI',Roboto,sans-serif;background:#ffffff;border:1px solid #cbd5e1;border-radius:12px;padding:20px;margin-top:8px;">
      <div style="background:linear-gradient(135deg,#0f172a,#1e3a8a);color:#fff;padding:18px 22px;border-radius:10px;margin-bottom:16px;">
        <div style="font-size:11px;text-transform:uppercase;letter-spacing:0.05em;color:#93c5fd;font-weight:700;">EPANET-AI &bull; Итоги обработки без консоли</div>
        <h3 style="margin:6px 0 4px 0;font-size:20px;color:#ffffff;">Очистка 4 целевых таблиц и создание новых файлов завершены</h3>
        <div style="font-size:13px;color:#cbd5e1;">
          Обработано городов: <b>{len(CLEAN_RESULTS)}</b> &bull;
          Удалено нулевых строк: <b>{total_removed}</b> &bull;
          Объём CSV: <b>{total_kb_before} КБ &rarr; {total_kb_after} КБ (&minus;{total_saved_pct}%)</b>
        </div>
      </div>
      <table style="width:100%;border-collapse:collapse;font-size:13px;color:#0f172a;">
        <thead>
          <tr style="background:#f8fafc;border-bottom:2px solid #cbd5e1;text-align:left;">
            <th style="padding:8px;">Город</th>
            <th style="padding:8px;text-align:right;">Узел (без Hтреб)</th>
            <th style="padding:8px;text-align:right;">Поток Насос (все 0)</th>
            <th style="padding:8px;text-align:right;">Потребители (Q=0)</th>
            <th style="padding:8px;text-align:right;">Потребление (все 0)</th>
            <th style="padding:8px;text-align:right;">Всего удалено</th>
            <th style="padding:8px;text-align:right;">Размер До &rarr; После</th>
            <th style="padding:8px;">Папка новых файлов</th>
          </tr>
        </thead>
        <tbody>{city_rows_html}</tbody>
      </table>
      <p style="margin:14px 0 0 0;font-size:12.5px;color:#475569;">
        Справочная формула скорости (Walski et al., <i>Advanced Water Distribution Modeling and Management</i>, p. 40, Eq. 2.9):
        <code>V = Q / A = 4Q / (&pi;D&sup2;)</code> (для <code>Q</code> в м&sup3;/ч и <code>D</code> в мм: <code>V [м/с] &approx; 353.67765 &middot; |Q| / D&sup2;</code>).
      </p>
    </div>
    """

    html_report_path = REPORTS_DIR / "epanet_ai_zero_clean_report.html"
    html_report_path.write_text(
        f"<!DOCTYPE html><html lang='ru'><head><meta charset='utf-8'><title>EPANET-AI Report</title></head>"
        f"<body style='background:#f1f5f9;padding:24px;'>{report_card_html}</body></html>",
        encoding="utf-8",
    )

    print(f"📄 JSON-отчёт сохранён : {json_report_path.resolve()}")
    print(f"🌐 HTML-отчёт сохранён : {html_report_path.resolve()}")
    display(HTML(report_card_html))
else:
    print("Нет обработанных городов — проверьте путь DATA_DIR в Клетке 1.")

📄 JSON-отчёт сохранён : D:\Zhenechka\code\EPANET_AI\artifacts_epanet_ai\reports\epanet_ai_zero_clean_report.json
🌐 HTML-отчёт сохранён : D:\Zhenechka\code\EPANET_AI\artifacts_epanet_ai\reports\epanet_ai_zero_clean_report.html


Город,Узел (без Hтреб),Поток Насос (все 0),Потребители (Q=0),Потребление (все 0),Всего удалено,Размер До → После,Папка новых файлов
Алматы,98006,0,47536,48226,193768,1673233.0 → 1331191.7 КБ (−20.44%),D:\Zhenechka\code\EPANET_AI\data_cleaned\Алматы
куляб,1218,16,1212,120,2566,31757.2 → 30760.2 КБ (−3.14%),D:\Zhenechka\code\EPANET_AI\data_cleaned\куляб
модель семей,21004,35,20927,292,42258,515728.8 → 512768.6 КБ (−0.57%),D:\Zhenechka\code\EPANET_AI\data_cleaned\модель семей
Орша,8451,44,2435,7983,18913,201477.9 → 175381.6 КБ (−12.95%),D:\Zhenechka\code\EPANET_AI\data_cleaned\Орша
тенино,316,15,320,320,971,13699.8 → 11335.0 КБ (−17.26%),D:\Zhenechka\code\EPANET_AI\data_cleaned\тенино
хорезмская,9878,365,9209,1871,21323,248958.4 → 232817.7 КБ (−6.48%),D:\Zhenechka\code\EPANET_AI\data_cleaned\хорезмская
